# 10 · Learned hybrid recommendation with LightFM

## Project context

This notebook follows the completed experiments in `07_hybrid_recommender.ipynb` and `09_reserved_slot_hybrid.ipynb`. Our current strongest reference is **implicit ALS alone**. The frozen **content-v1** representation uses MovieLens genres + release decade. **TMDB-v2** extends that base representation with TMDB genres, keywords, overview text, director, and cast.

The earlier experiments found:

- Score-percentile blends of ALS and content substantially reduced overall full-catalogue NDCG@10.
- Reserved-slot recommendations increased rare-movie exposure, but no tested candidate passed the project's 98% overall-NDCG retention floor.
- Profile-based content similarity did not retrieve any of the held-out strict-cold targets into the top 10 in the tested validation window.

We are now changing the learning architecture, not continuing to tune manual blending weights.

## What LightFM changes

LightFM learns latent embeddings for user/item identities and can also learn embeddings for item features. In the metadata-enhanced model, an item's representation is a weighted combination of its identity embedding and its metadata-feature embeddings. This lets interaction data teach the model which metadata patterns are useful for recommendation, rather than assuming that content similarity alone is the correct ranking signal.

Official references:

- [LightFM project and installation](https://github.com/lyst/lightfm)
- [LightFM quickstart](https://github.com/lyst/lightfm/blob/master/doc/quickstart.rst)
- [LightFM API documentation](https://making.lyst.com/lightfm/docs/lightfm.html)

## Candidates

1. **ALS-only** — existing collaborative reference, refitted on training data only.
2. **LightFM identity-only** — collaborative model with user/item identity features and no movie metadata.
3. **LightFM + content-v1** — item identity plus genres and release decade.
4. **LightFM + TMDB-v2** — item identity plus the saved TMDB-enriched content representation.

The identity-only LightFM comparison is important: it isolates whether adding metadata helps *within LightFM*, rather than confusing a change in algorithm with a benefit from the features.

## 1 · Experimental protocol and decision rules

### Fixed data and target definition

- Dataset: MovieLens `sample_10pct` and its established per-user temporal `train`, `val`, and `test` splits.
- Positive interaction / relevance: rating `>= 4.0`.
- LightFM loss: WARP, a pairwise ranking loss for implicit positive interactions.
- Catalogue: 17,241 modelled movies; item/user indices must align with `recutils.load_split`.
- Validation target window: positive validation interactions at or after `2016-01-01 00:00:00 UTC`, matching notebook 07/09.
- Validation tail pool: movies with fewer than 20 positive interactions in the original `train` split; strict-cold = 0, warm-tail = 1–19.
- View A: all catalogue movies compete for the top 10; only target relevance is restricted when calculating tail/cold/warm-tail metrics.
- Seen historical movies are never recommended.

### Controlled LightFM comparison

All LightFM candidates use the same initial hyperparameters: 32 components, WARP loss, learning rate `0.05`, `15` epochs, and fixed seed. These are a controlled first experiment, not a claim that the hyperparameters are optimal. We do not run a large grid in this notebook.

Metadata row weighting is fixed before looking at validation results: for movies with available features, identity features carry half of the item-feature weight and the normalized metadata group carries the other half. For movies with no metadata features, identity weight remains 1.0. This prevents the large number of text columns from winning merely because their raw weights sum to more than the identity feature.

### Decision rules

- Primary model-development question: does metadata-enhanced LightFM improve tail View A NDCG@10 versus identity-only LightFM? Use paired user-level 95% bootstrap intervals; a point-estimate increase alone is not enough evidence.
- The original 98% overall-NDCG guardrail applies to considering LightFM as a wholesale replacement for ALS.
- The later selective-reranking experiment is a separate intervention. Its 500-rank gate was selected using validation analysis and frozen before the final holdout evaluation; the 1,000-rank gate is reported only as a pre-specified higher-discovery comparison.
- The final holdout comparison is recorded in Section 13. Do not retune the gate, change relevance rules, or otherwise make model-selection decisions based on those test results.

### Important limitation

The split is an established per-user temporal split, while each evaluation target window is filtered to 2016 onward to match the previous full-catalogue report. The cutoff aligns evaluated targets; it does **not** prove every training interaction globally precedes 2016. The split audit found no user–movie pair overlap and nondecreasing per-user timestamp bounds across train, validation, and test, but timestamp ties exist for some users.


## 2 · Environment setup

Run the notebook from the project's existing `.venv` / Python environment. LightFM is an additional dependency.

If the import below fails, stop and install it in the same environment used by this notebook:

```powershell
python -m pip install lightfm
```

If installation fails on Windows due to compilation/build dependencies, send the complete installation error rather than changing Python versions or installing random build tools. The LightFM project also documents Conda installation (`conda install -c conda-forge lightfm`).

This notebook uses only saved local feature matrices; **it makes no TMDB API calls**.

In [1]:
# Setup and dependency checks
import os
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import lightfm
    from lightfm import LightFM
except ImportError as exc:
    raise ImportError(
        "LightFM is not installed in this notebook's Python environment. "
        "In the project's active environment run: python -m pip install lightfm. "
        "If it fails to build on Windows, send the complete error output."
    ) from exc

# Find the project root, then use the same notebooks/ working directory convention
# used by the existing project notebooks and scripts/recutils.py.
_START = Path.cwd().resolve()
_PROJECT_ROOT = None
for _p in [_START, *_START.parents]:
    if (_p / "scripts" / "recutils.py").exists() and (_p / "data").exists():
        _PROJECT_ROOT = _p
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing scripts/recutils.py and data/. "
        "Place this notebook in the project's notebooks/ folder and reopen it there."
    )
_NOTEBOOK_DIR = _PROJECT_ROOT / "notebooks"
if _NOTEBOOK_DIR.exists() and Path.cwd().resolve() != _NOTEBOOK_DIR.resolve():
    os.chdir(_NOTEBOOK_DIR)
sys.path.insert(0, str(_PROJECT_ROOT / "scripts"))

from recutils import load_split

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42
CUTOFF_TS = int(pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp())
RETENTION_FRACTION = 0.98
N_BOOT = 1000
N_TIE_SEEDS = 10
SCORE_BATCH_SIZE = 64
NUM_THREADS = 1  # reproducibility first; increase only if runtime requires it

# First controlled LightFM configuration. Keep this fixed across the three models.
LIGHTFM_COMPONENTS = 32
LIGHTFM_EPOCHS = 15
LIGHTFM_LEARNING_RATE = 0.05
LIGHTFM_ITEM_ALPHA = 1e-6
LIGHTFM_USER_ALPHA = 1e-6
LIGHTFM_MAX_SAMPLED = 10

print("Project root:", _PROJECT_ROOT)
print("LightFM version:", getattr(lightfm, "__version__", "version unavailable"))
print("LightFM config:", {
    "loss": "warp",
    "no_components": LIGHTFM_COMPONENTS,
    "epochs": LIGHTFM_EPOCHS,
    "learning_rate": LIGHTFM_LEARNING_RATE,
    "item_alpha": LIGHTFM_ITEM_ALPHA,
    "user_alpha": LIGHTFM_USER_ALPHA,
    "max_sampled": LIGHTFM_MAX_SAMPLED,
    "num_threads": NUM_THREADS,
})

Project root: /mnt/c/Users/Lenovo/Documents/movie-recommender
LightFM version: 1.17
LightFM config: {'loss': 'warp', 'no_components': 32, 'epochs': 15, 'learning_rate': 0.05, 'item_alpha': 1e-06, 'user_alpha': 1e-06, 'max_sampled': 10, 'num_threads': 1}


## 3 · Load the established split and frozen feature artifacts

The model-development section initially loads `train` and `val` only, so the test split remains unused during model comparison and reranking-threshold selection. Feature matrices must be in the exact same internal item order as the split's `item_map`.

The validation target window is constructed once and reused for every model and every evaluation scope. After the 500-rank gate is frozen and the two selected models are refitted on `train + val`, Section 13 loads the existing test split for the final holdout comparison. No threshold tuning should follow that comparison.


In [2]:
# Load train/validation data only; do not read S["test"].
S = load_split(NAME)
train = S["train"].copy()
val = S["val"].copy()
item_map = S["item_map"].sort_values("i").reset_index(drop=True)
user_map = S["user_map"].sort_values("u").reset_index(drop=True)
n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

feature_dir = _PROJECT_ROOT / "data" / "interim" / "hybrid_features"
tmdb_dir = _PROJECT_ROOT / "data" / "processed" / NAME
v1_path = feature_dir / "content_v1.npz"
tmdb_path = tmdb_dir / "tmdb_content_v2.npz"
tmdb_index_path = tmdb_dir / "tmdb_content_v2_index.parquet"

for _path in (v1_path, tmdb_path, tmdb_index_path):
    if not _path.exists():
        raise FileNotFoundError(
            f"Required local artifact not found: {_path}\n"
            "Generate it in the earlier content/TMDB notebooks first."
        )

X_v1 = sp.load_npz(v1_path).tocsr().astype(np.float32)
X_tmdb_v2 = sp.load_npz(tmdb_path).tocsr().astype(np.float32)
tmdb_item_index = pd.read_parquet(tmdb_index_path).sort_values("i").reset_index(drop=True)

assert len(train) > 0 and len(val) > 0
assert item_map["u"].is_unique if "u" in item_map.columns else True
assert item_map["i"].is_unique and item_map["movieId"].is_unique
assert user_map["u"].is_unique and user_map["userId"].is_unique
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert X_v1.shape == (n_items, 31), f"Unexpected content-v1 shape: {X_v1.shape}"
assert X_tmdb_v2.shape[0] == n_items, f"Unexpected TMDB-v2 shape: {X_tmdb_v2.shape}"
assert np.array_equal(tmdb_item_index["i"].to_numpy(), item_map["i"].to_numpy())
assert np.array_equal(tmdb_item_index["movieId"].to_numpy(), item_map["movieId"].to_numpy())
assert np.isfinite(X_v1.data).all() and np.isfinite(X_tmdb_v2.data).all()

# Seen mask includes ALL training ratings; only recommendations should be unseen.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(dtype=np.int64), train["i"].to_numpy(dtype=np.int64)),
    ),
    shape=(n_users, n_items),
)
seen_train.sum_duplicates()
seen_train.data[:] = 1.0

train_positive = train.loc[train["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
n_pos = np.bincount(train_positive["i"].to_numpy(dtype=np.int64), minlength=n_items)
tail_mask = n_pos < 20
strict_cold_mask = n_pos == 0
warm_tail_mask = (n_pos >= 1) & (n_pos < 20)
item_has_positive_history = n_pos > 0

# The established fallback is based on all training interactions, not only positive ones.
popularity_scores = np.bincount(
    train["i"].to_numpy(dtype=np.int64), minlength=n_items
).astype(np.float64)
u_has_pos = np.bincount(
    train_positive["u"].to_numpy(dtype=np.int64), minlength=n_users
) > 0

assert "timestamp" in val.columns, "Validation split lacks timestamp column."
val_ts = pd.to_numeric(val["timestamp"], errors="raise")
val_window = val.loc[
    (val["rating"] >= REL) & (val_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

def relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

item_idx_val = val_window["i"].to_numpy(dtype=np.int64)
relevant_by_scope = {
    "overall": relevant_dict(val_window),
    "tail": relevant_dict(val_window.loc[tail_mask[item_idx_val]]),
    "strict_cold": relevant_dict(val_window.loc[strict_cold_mask[item_idx_val]]),
    "warm_tail": relevant_dict(val_window.loc[warm_tail_mask[item_idx_val]]),
}
scope_users = {
    scope: np.asarray(sorted(rel.keys()), dtype=np.int64)
    for scope, rel in relevant_by_scope.items()
}
eval_users = scope_users["overall"]

# Guard against accidental train/validation pair overlap.
_target_u = val_window["u"].to_numpy(dtype=np.int64)
_target_i = val_window["i"].to_numpy(dtype=np.int64)
assert not np.asarray(seen_train[_target_u, _target_i]).ravel().any(), (
    "Some validation target pairs are present in training history."
)

print(f"Users: {n_users:,} | Catalogue movies: {n_items:,}")
print(f"Train interactions: {len(train):,} | Positive train pairs (rating >= {REL}): {len(train_positive):,}")
print(f"Validation rows: {len(val):,} | Evaluated post-2016 positive pairs: {len(val_window):,}")
print(f"content-v1: {X_v1.shape}, nnz={X_v1.nnz:,}")
print(f"TMDB-v2:    {X_tmdb_v2.shape}, nnz={X_tmdb_v2.nnz:,}")
print(f"Tail (<20 positive train pairs): {int(tail_mask.sum()):,} movies")
print(f"Strict cold (0 positive train pairs): {int(strict_cold_mask.sum()):,} movies")
print(f"Warm tail (1–19 positive train pairs): {int(warm_tail_mask.sum()):,} movies")
for _scope, _rel in relevant_by_scope.items():
    print(f"{_scope:12s}: {sum(map(len, _rel.values())):,} relevant pairs, {len(_rel):,} users")
print("Alignment and leakage checks passed. Test split not accessed.")

Users: 20,093 | Catalogue movies: 17,241
Train interactions: 2,919,680 | Positive train pairs (rating >= 4.0): 1,445,945
Validation rows: 137,197 | Evaluated post-2016 positive pairs: 25,406
content-v1: (17241, 31), nnz=53,959
TMDB-v2:    (17241, 91328), nnz=873,444
Tail (<20 positive train pairs): 11,597 movies
Strict cold (0 positive train pairs): 1,095 movies
Warm tail (1–19 positive train pairs): 10,502 movies
overall     : 25,406 relevant pairs, 5,661 users
tail        : 1,585 relevant pairs, 948 users
strict_cold : 52 relevant pairs, 50 users
warm_tail   : 1,533 relevant pairs, 931 users
Alignment and leakage checks passed. Test split not accessed.


## 4 · Build interactions and item feature matrices

### Interactions

WARP needs observed positive user–item interactions. We use training ratings `>= 4.0`, exactly matching the project's definition of a positive. Missing user–movie pairs are treated as unobserved candidates from which the loss samples negatives; they are not written into the interaction matrix as explicit dislike labels.

### Identity versus metadata

- Identity-only: one one-hot item identity feature per movie.
- Metadata-enhanced: the same one-hot identity block plus the selected content representation.
- User features are identity-only in every model; this experiment isolates item metadata.

Before concatenation, each movie's metadata row is L1-normalized. For movies with metadata, the total row weight is divided equally between identity and metadata. This is a fixed, interpretable starting point, not a tuned feature-weight sweep.

In [3]:
# LightFM's implicit-positive interaction matrix.
interactions = sp.coo_matrix(
    (
        np.ones(len(train_positive), dtype=np.float32),
        (
            train_positive["u"].to_numpy(dtype=np.int32),
            train_positive["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
    dtype=np.float32,
)
interactions.sum_duplicates()
interactions.eliminate_zeros()
assert interactions.nnz == len(train_positive)

# Explicit identity features keep the identity-only and hybrid designs conceptually aligned.
I_items = sp.eye(n_items, format="csr", dtype=np.float32)

def make_identity_plus_metadata_features(X_side, metadata_mass=0.5):
    """Return item features with fixed identity/metadata mass and safe empty rows."""
    X_side = X_side.tocsr().astype(np.float32)
    assert X_side.shape[0] == n_items

    X_side = normalize(X_side, norm="l1", axis=1, copy=True).tocsr().astype(np.float32)
    has_metadata = np.asarray(X_side.getnnz(axis=1) > 0).ravel()

    identity_weights = np.where(has_metadata, 1.0 - metadata_mass, 1.0).astype(np.float32)
    identity_block = sp.diags(identity_weights, format="csr") @ I_items
    metadata_block = X_side.multiply(metadata_mass).tocsr()

    features = sp.hstack([identity_block, metadata_block], format="csr", dtype=np.float32)
    row_sums = np.asarray(features.sum(axis=1)).ravel()
    assert np.allclose(row_sums, 1.0, atol=1e-5), (
        f"Item-feature row weights must sum to one; min={row_sums.min()}, max={row_sums.max()}"
    )
    return features

item_features_identity = I_items
item_features_v1 = make_identity_plus_metadata_features(X_v1, metadata_mass=0.5)
item_features_tmdb_v2 = make_identity_plus_metadata_features(X_tmdb_v2, metadata_mass=0.5)

item_feature_sets = {
    "LightFM_identity": item_features_identity,
    "LightFM_content_v1": item_features_v1,
    "LightFM_TMDB_v2": item_features_tmdb_v2,
}

print("Positive interaction matrix:", interactions.shape, "nnz=", f"{interactions.nnz:,}")
for name, feature_matrix in item_feature_sets.items():
    print(
        f"{name:20s} | shape={feature_matrix.shape} | nnz={feature_matrix.nnz:,} | "
        f"item rows with at least one feature={int((feature_matrix.getnnz(axis=1) > 0).sum()):,}"
    )
print("Feature matrices constructed; no feature weights were tuned on validation.")

Positive interaction matrix: (20093, 17241) nnz= 1,445,945
LightFM_identity     | shape=(17241, 17241) | nnz=17,241 | item rows with at least one feature=17,241
LightFM_content_v1   | shape=(17241, 17272) | nnz=71,200 | item rows with at least one feature=17,241
LightFM_TMDB_v2      | shape=(17241, 108569) | nnz=890,685 | item rows with at least one feature=17,241
Feature matrices constructed; no feature weights were tuned on validation.


## 5 · Fit the ALS-only reference on training data

We refit the same implicit ALS setup used in notebook 09, using positive training pairs only. The model is evaluated on the same post-2016 validation window as LightFM. This costs roughly 30–40 seconds in the previous run, but it gives us per-user reference recommendations for paired comparisons.

In [4]:
# Train-only implicit ALS reference, copied from the validated project setup.
def _implicit_half_step(M, Y, lam):
    """One ALS half-step; M stores confidence-minus-one for observed positives."""
    n_rows = M.shape[0]
    k = Y.shape[1]
    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)
    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]
        if start == end:
            continue
        factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]
        A = YtY + (factors.T * confidence_minus_one) @ factors + reg
        b = factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)
    return out


def fit_implicit_als_train_only(df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED):
    d = df.loc[df["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(dtype=np.int64), d["i"].to_numpy(dtype=np.int64)),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)
    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)
        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(f"ALS iteration {iteration:02d}/{n_iter} | elapsed {time.time() - start_time:.1f}s")
    return {"P": P.astype(np.float32), "Q": Q.astype(np.float32)}

als_model = fit_implicit_als_train_only(
    train, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED
)
P_als, Q_als = als_model["P"], als_model["Q"]
item_has_als_factors = np.abs(Q_als).sum(axis=1) > 0
assert P_als.shape == (n_users, 64) and Q_als.shape == (n_items, 64)
assert np.isfinite(P_als).all() and np.isfinite(Q_als).all()
assert int((~item_has_als_factors).sum()) == int(strict_cold_mask.sum())

print("ALS refit complete.")
print("Users without positive history:", int((~u_has_pos).sum()))
print("Items without learned ALS factors:", int((~item_has_als_factors).sum()))

ALS iteration 01/10 | elapsed 10.8s
ALS iteration 02/10 | elapsed 17.1s
ALS iteration 04/10 | elapsed 34.1s
ALS iteration 06/10 | elapsed 62.2s
ALS iteration 08/10 | elapsed 73.8s
ALS iteration 10/10 | elapsed 84.6s
ALS refit complete.
Users without positive history: 34
Items without learned ALS factors: 1095


## 6 · Define and fit the three LightFM models

We use WARP because this experiment is about ranking relevant movies near the top rather than minimizing rating-prediction error. Every model receives the same positive interactions, loss, dimensionality, epoch count, regularization and seed. The metadata models differ only in the item-feature matrix.

These fits can take several minutes. Run each cell once. If a model fails during fitting, stop and share the complete traceback before changing configurations.

In [5]:
def fit_lightfm_candidate(name, item_features):
    start_time = time.time()
    model = LightFM(
        no_components=LIGHTFM_COMPONENTS,
        loss="warp",
        learning_rate=LIGHTFM_LEARNING_RATE,
        item_alpha=LIGHTFM_ITEM_ALPHA,
        user_alpha=LIGHTFM_USER_ALPHA,
        max_sampled=LIGHTFM_MAX_SAMPLED,
        random_state=SEED,
    )
    model.fit(
        interactions,
        item_features=item_features,
        epochs=LIGHTFM_EPOCHS,
        num_threads=NUM_THREADS,
        verbose=True,
    )
    print(f"{name} fit complete in {(time.time() - start_time) / 60:.1f} min")
    print("User embeddings:", model.user_embeddings.shape)
    print("Item feature embeddings:", model.item_embeddings.shape)
    return model

lightfm_models = {}

In [6]:
from scipy.sparse import coo_matrix
from lightfm import LightFM

interactions_test = coo_matrix(
    ([1.0, 1.0, 1.0, 1.0], ([0, 0, 1, 1], [0, 1, 0, 2])),
    shape=(2, 3),
    dtype="float32",
)

model_test = LightFM(no_components=5, loss="warp")
model_test.fit(interactions_test, epochs=2, num_threads=1)

print("Minimal training successful")

Minimal training successful


In [7]:
# 6a · Identity-only LightFM
lightfm_models["LightFM_identity"] = fit_lightfm_candidate(
    "LightFM_identity",
    item_features_identity,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_identity fit complete in 0.4 min
User embeddings: (20093, 32)
Item feature embeddings: (17241, 32)


In [8]:
# 6b · LightFM with the frozen content-v1 metadata (genres + release decade)
lightfm_models["LightFM_content_v1"] = fit_lightfm_candidate(
    "LightFM_content_v1",
    item_features_v1,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_content_v1 fit complete in 0.6 min
User embeddings: (20093, 32)
Item feature embeddings: (17272, 32)


In [9]:
# 6c · LightFM with the TMDB-enriched content-v2 representation
lightfm_models["LightFM_TMDB_v2"] = fit_lightfm_candidate(
    "LightFM_TMDB_v2",
    item_features_tmdb_v2,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_TMDB_v2 fit complete in 8.5 min
User embeddings: (20093, 32)
Item feature embeddings: (108569, 32)


## 7 · Verify scoring and prepare full-catalogue scorers

LightFM combines feature embeddings to obtain each item's latent representation and bias. The following scorer uses those representations to compute a batch of user-by-catalogue scores efficiently. It is checked against the official `model.predict` method on a few pairs before full validation scoring.

In [10]:
def make_lightfm_scorer(model, item_features):
    # These products are small dense matrices: (n_items, n_components) plus one bias per item.
    item_vectors = np.asarray(item_features @ model.item_embeddings, dtype=np.float32)
    item_bias = np.asarray(item_features @ model.item_biases).reshape(-1).astype(np.float32)
    user_vectors = np.asarray(model.user_embeddings, dtype=np.float32)
    user_bias = np.asarray(model.user_biases, dtype=np.float32)

    assert item_vectors.shape == (n_items, LIGHTFM_COMPONENTS)
    assert user_vectors.shape == (n_users, LIGHTFM_COMPONENTS)
    assert item_bias.shape == (n_items,) and user_bias.shape == (n_users,)

    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int32)
        scores = user_vectors[ub] @ item_vectors.T
        scores += user_bias[ub, None]
        scores += item_bias[None, :]
        return scores.astype(np.float32, copy=False)

    return score_batch


def make_als_scorer(P, Q):
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int64)
        return (P[ub] @ Q.T).astype(np.float32, copy=False)
    return score_batch

score_functions = {
    "ALS_only": make_als_scorer(P_als, Q_als),
}
for model_name, model in lightfm_models.items():
    score_functions[model_name] = make_lightfm_scorer(
        model, item_feature_sets[model_name]
    )

# Smoke-test direct matrix scoring against LightFM's official predict() on each fitted model.
smoke_user = int(eval_users[0])
smoke_items = np.asarray([0, min(1, n_items - 1), min(17, n_items - 1)], dtype=np.int32)
for model_name, model in lightfm_models.items():
    feature_matrix = item_feature_sets[model_name]
    official = model.predict(
        user_ids=np.full(len(smoke_items), smoke_user, dtype=np.int32),
        item_ids=smoke_items,
        item_features=feature_matrix,
        num_threads=NUM_THREADS,
    )
    direct = score_functions[model_name](np.asarray([smoke_user]))[0, smoke_items]
    assert np.allclose(official, direct, rtol=1e-4, atol=1e-5), (
        f"Scorer mismatch for {model_name}: official={official}, direct={direct}"
    )
    print(f"{model_name}: official-predict scorer check passed")

print("All scorers ready:", list(score_functions))

LightFM_identity: official-predict scorer check passed
LightFM_content_v1: official-predict scorer check passed
LightFM_TMDB_v2: official-predict scorer check passed
All scorers ready: ['ALS_only', 'LightFM_identity', 'LightFM_content_v1', 'LightFM_TMDB_v2']


## 8 · Rank the full catalogue once per model

Recommendations are generated for the users who have at least one relevant target in the overall validation window. The same final recommendation list for each user and tie-breaking seed is used to compute overall, tail, strict-cold and warm-tail metrics.

Rules:

- Remove every item the user rated in training, regardless of rating.
- ALS excludes movies without learned ALS factors, matching the established ALS policy.
- Users without positive training history use the existing popularity fallback for every model.
- LightFM metadata models may score zero-positive-history movies from shared item features; this is one of the behaviours being tested.
- A tiny noise term only resolves nearly exact score ties; it does not constitute score blending.

In [11]:
def popularity_fallback_scores(user_id):
    scores = popularity_scores.copy()
    scores[seen_train[user_id].indices] = -np.inf
    return scores


def top_k_from_scores(scores, user_id, model_name, tie_seed, k=K):
    row = np.asarray(scores, dtype=np.float64).copy()

    # Users with no positive train interactions receive the agreed fallback list.
    if not u_has_pos[user_id]:
        row = popularity_fallback_scores(user_id)
    elif model_name == "ALS_only":
        row[~item_has_als_factors] = -np.inf

    # Never recommend any movie that appeared in the user's training history.
    row[seen_train[user_id].indices] = -np.inf
    eligible = np.isfinite(row)
    if int(eligible.sum()) < k:
        raise RuntimeError(f"Only {int(eligible.sum())} eligible movies for user {user_id}")

    # Seeded per-user tie-break. The same noise vector is used across models for a given
    # user/seed; epsilon is tiny relative to the model score scale.
    tie_rng = np.random.default_rng(np.random.SeedSequence([SEED, int(tie_seed), int(user_id)]))
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(row[eligible]))))
    adjusted = row.copy()
    adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

    candidates = np.argpartition(adjusted, -k)[-k:]
    return candidates[np.argsort(adjusted[candidates])[::-1]].astype(np.int32)

seen_sets = {int(u): set(seen_train[int(u)].indices.tolist()) for u in eval_users}
recommendations = {}
start_all = time.time()

for model_name, score_fn in score_functions.items():
    start_model = time.time()
    recs_by_seed = {seed: {} for seed in range(N_TIE_SEEDS)}
    print(f"\nScoring and ranking {model_name} ...")

    for start in range(0, len(eval_users), SCORE_BATCH_SIZE):
        user_batch = eval_users[start:start + SCORE_BATCH_SIZE]
        batch_scores = score_fn(user_batch)
        if batch_scores.shape != (len(user_batch), n_items):
            raise ValueError(
                f"{model_name} returned shape {batch_scores.shape}; "
                f"expected {(len(user_batch), n_items)}"
            )
        if not np.isfinite(batch_scores).all():
            raise ValueError(f"Non-finite scores from {model_name}")

        for row_idx, user_id in enumerate(user_batch):
            raw_scores = batch_scores[row_idx]
            for tie_seed in range(N_TIE_SEEDS):
                recs_by_seed[tie_seed][int(user_id)] = top_k_from_scores(
                    raw_scores, int(user_id), model_name, tie_seed
                )

        processed = min(start + len(user_batch), len(eval_users))
        if processed == len(user_batch) or processed % 512 < SCORE_BATCH_SIZE or processed == len(eval_users):
            print(f"  ranked {processed:,}/{len(eval_users):,} users")

    recommendations[model_name] = recs_by_seed
    print(f"{model_name} completed in {(time.time() - start_model) / 60:.1f} min")
    gc.collect()

print(f"\nAll ranking completed in {(time.time() - start_all) / 60:.1f} min")

# Recommendation-list integrity checks, including no seen items or duplicates.
for model_name, by_seed in recommendations.items():
    for tie_seed, user_recs in by_seed.items():
        assert len(user_recs) == len(eval_users)
        for user_id, slate in user_recs.items():
            assert len(slate) == K and len(set(map(int, slate))) == K
            assert seen_sets[user_id].isdisjoint(map(int, slate)), (model_name, tie_seed, user_id)
    print(f"{model_name}: slate integrity checks passed across {N_TIE_SEEDS} seeds")


Scoring and ranking ALS_only ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
ALS_only completed in 0.4 min

Scoring and ranking LightFM_identity ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
LightFM_identity completed in 0.4 min

Scoring and ranking LightFM_content_v1 ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,

## 9 · Calculate user-level metrics for every evaluation scope

We calculate per-user NDCG@10, Recall@10, HitRate@10 and hit count. Metrics are computed separately for overall, tail, strict-cold and warm-tail target sets. For a slice-specific metric, the recommendation candidate pool is still the full catalogue; only the relevant target set is restricted to that slice.

Per-user metrics are averaged over the fixed tie-breaking seeds before bootstrap resampling users.

In [12]:
discounts = 1.0 / np.log2(np.arange(2, K + 2, dtype=np.float64))


def metrics_for_slate(slate, relevant_items):
    relevant_items = set(relevant_items)
    hits = np.asarray([int(item) in relevant_items for item in slate[:K]], dtype=np.float64)
    recall = float(hits.sum() / len(relevant_items)) if relevant_items else 0.0
    idcg_len = min(len(relevant_items), K)
    idcg = float(discounts[:idcg_len].sum()) if idcg_len else 0.0
    ndcg = float((hits * discounts[:len(hits)]).sum() / idcg) if idcg > 0 else 0.0
    return {
        "ndcg@10": ndcg,
        "recall@10": recall,
        "hit_rate@10": float(hits.sum() > 0),
        "hit_count@10": float(hits.sum()),
    }


user_metrics = {model_name: {} for model_name in recommendations}
absolute_metric_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    print(f"Evaluating {scope_name}: {len(users_scope):,} users")

    for model_name in recommendations:
        per_seed_frames = []
        for tie_seed in range(N_TIE_SEEDS):
            values = []
            recs = recommendations[model_name][tie_seed]
            for user_id in users_scope:
                values.append(metrics_for_slate(recs[int(user_id)], rel_dict[int(user_id)]))
            frame = pd.DataFrame(values, index=users_scope)
            frame.index.name = "u"
            per_seed_frames.append(frame)

        averaged = sum(per_seed_frames) / float(N_TIE_SEEDS)
        averaged.index.name = "u"
        user_metrics[model_name][scope_name] = averaged

        for metric in ("ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"):
            absolute_metric_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": float(averaged[metric].mean()),
            })

absolute_metrics = pd.DataFrame(absolute_metric_rows)
print("\nVALIDATION METRIC POINT ESTIMATES")
display(
    absolute_metrics.loc[absolute_metrics["metric"].isin(["ndcg@10", "recall@10", "hit_rate@10"])]
    .pivot_table(index=["scope", "metric"], columns="model", values="mean")
    .round(6)
)

# Sanity check: every model has the same users in every scope.
for scope_name, rel_dict in relevant_by_scope.items():
    for model_name in user_metrics:
        assert user_metrics[model_name][scope_name].index.tolist() == sorted(rel_dict)

Evaluating overall: 5,661 users
Evaluating tail: 948 users
Evaluating strict_cold: 50 users
Evaluating warm_tail: 931 users

VALIDATION METRIC POINT ESTIMATES


model                    ALS_only  LightFM_TMDB_v2  LightFM_content_v1  \
scope       metric                                                       
overall     hit_rate@10  0.302067         0.254902            0.251546   
            ndcg@10      0.081462         0.063340            0.062729   
            recall@10    0.106220         0.083822            0.082748   
strict_cold hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   
tail        hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   
warm_tail   hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   

model                    LightFM_identity  
scope       metric                         
overall     hit_rate@10          0.255255  
            ndcg@10              0.063860  
            recall@10            0.082599  
strict_cold hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000  
tail        hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000  
warm_tail   hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000

## 10 · Paired user-level bootstrap confidence intervals

For each comparison we bootstrap the *paired per-user difference*, resampling the same users for both models. This is more informative than comparing two independent intervals because each candidate is evaluated on the same users and targets.

Primary comparison: metadata-enhanced LightFM versus identity-only LightFM on tail View A NDCG@10. Secondary comparisons: each LightFM model versus ALS, plus strict-cold and warm-tail results. The intervals quantify validation uncertainty; they do not undo the fact that multiple model variants were considered.

In [13]:
def paired_bootstrap(delta_values, draw_indices):
    delta_values = np.asarray(delta_values, dtype=np.float64)
    point = float(delta_values.mean())
    boot_means = delta_values[draw_indices].mean(axis=1)
    low, high = np.percentile(boot_means, [2.5, 97.5])
    return point, float(low), float(high)

bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = {}
for scope_name, rel_dict in relevant_by_scope.items():
    n_scope = len(rel_dict)
    bootstrap_indices[scope_name] = bootstrap_rng.integers(
        0, n_scope, size=(N_BOOT, n_scope), endpoint=False
    )

comparisons = [
    ("ALS_only", "LightFM_identity"),
    ("ALS_only", "LightFM_content_v1"),
    ("ALS_only", "LightFM_TMDB_v2"),
    ("LightFM_identity", "LightFM_content_v1"),
    ("LightFM_identity", "LightFM_TMDB_v2"),
]
metrics_to_compare = ("ndcg@10", "recall@10", "hit_rate@10")
paired_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]

    for reference_name, candidate_name in comparisons:
        reference_frame = user_metrics[reference_name][scope_name].loc[users_scope]
        candidate_frame = user_metrics[candidate_name][scope_name].loc[users_scope]

        for metric in metrics_to_compare:
            reference_values = reference_frame[metric].to_numpy(dtype=np.float64)
            candidate_values = candidate_frame[metric].to_numpy(dtype=np.float64)
            delta = candidate_values - reference_values
            point, low, high = paired_bootstrap(delta, draws)
            paired_rows.append({
                "scope": scope_name,
                "reference": reference_name,
                "candidate": candidate_name,
                "metric": metric,
                "users": len(users_scope),
                "reference_mean": float(reference_values.mean()),
                "candidate_mean": float(candidate_values.mean()),
                "paired_delta": point,
                "delta_ci95_low": low,
                "delta_ci95_high": high,
            })

paired_ci_results = pd.DataFrame(paired_rows)
print("PAIRED VALIDATION COMPARISONS")
display(
    paired_ci_results.loc[
        paired_ci_results["metric"].isin(["ndcg@10", "recall@10"])
    ].round(6)
)

primary = paired_ci_results.loc[
    (paired_ci_results["scope"] == "tail")
    & (paired_ci_results["metric"] == "ndcg@10")
    & (paired_ci_results["reference"] == "LightFM_identity")
]
print("\nPRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?")
display(primary.round(6))

PAIRED VALIDATION COMPARISONS


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
0,overall,ALS_only,LightFM_identity,ndcg@10,5661,0.081462,0.063860,-0.017602,-0.021017,-0.014213
1,overall,ALS_only,LightFM_identity,recall@10,5661,0.106220,0.082599,-0.023621,-0.028082,-0.018911
3,overall,ALS_only,LightFM_content_v1,ndcg@10,5661,0.081462,0.062729,-0.018733,-0.022402,-0.015505
4,overall,ALS_only,LightFM_content_v1,recall@10,5661,0.106220,0.082748,-0.023472,-0.028381,-0.018753
6,overall,ALS_only,LightFM_TMDB_v2,ndcg@10,5661,0.081462,0.063340,-0.018122,-0.021618,-0.014545
7,overall,ALS_only,LightFM_TMDB_v2,recall@10,5661,0.106220,0.083822,-0.022398,-0.027485,-0.017533
9,overall,LightFM_identity,LightFM_content_v1,ndcg@10,5661,0.063860,0.062729,-0.001132,-0.003558,0.001309
10,overall,LightFM_identity,LightFM_content_v1,recall@10,5661,0.082599,0.082748,0.000149,-0.003417,0.003487
12,overall,LightFM_identity,LightFM_TMDB_v2,ndcg@10,5661,0.063860,0.063340,-0.000520,-0.002743,0.001482
13,overall,LightFM_identity,LightFM_TMDB_v2,recall@10,5661,0.082599,0.083822,0.001223,-0.002157,0.004525



PRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
24,tail,LightFM_identity,LightFM_content_v1,ndcg@10,948,0.0,0.0,0.0,0.0,0.0
27,tail,LightFM_identity,LightFM_TMDB_v2,ndcg@10,948,0.0,0.0,0.0,0.0,0.0


## 11 · Candidate decision table and reports

The 98% guardrail uses the ALS-only overall View A NDCG@10 **from this exact run**, not a hard-coded historical number. We report both the point-estimate pass/fail and a paired interval for the guardrail margin `G = candidate overall NDCG@10 − 0.98 × ALS overall NDCG@10`.

A LightFM model can still teach us that metadata helps LightFM while failing the overall ALS guardrail. In that case it is a useful experimental result, but it is not selected as the project's replacement recommender under the current rule.

In [14]:
als_overall_frame = user_metrics["ALS_only"]["overall"]
als_overall_ndcg_values = als_overall_frame["ndcg@10"].to_numpy(dtype=np.float64)
als_overall_ndcg = float(als_overall_ndcg_values.mean())
retention_floor = RETENTION_FRACTION * als_overall_ndcg

# Build a concise table with overall, tail, cold and warm-tail point estimates.
decision_rows = []
for model_name in ("ALS_only", "LightFM_identity", "LightFM_content_v1", "LightFM_TMDB_v2"):
    overall = user_metrics[model_name]["overall"]
    tail = user_metrics[model_name]["tail"]
    cold = user_metrics[model_name]["strict_cold"]
    warm = user_metrics[model_name]["warm_tail"]

    overall_ndcg = float(overall["ndcg@10"].mean())
    overall_recall = float(overall["recall@10"].mean())
    guard_low, guard_high = np.nan, np.nan
    guard_point = np.nan
    guard_pass = True if model_name == "ALS_only" else bool(overall_ndcg >= retention_floor)

    if model_name != "ALS_only":
        candidate_overall_values = overall["ndcg@10"].to_numpy(dtype=np.float64)
        guard_margin = candidate_overall_values - RETENTION_FRACTION * als_overall_ndcg_values
        guard_point, guard_low, guard_high = paired_bootstrap(
            guard_margin, bootstrap_indices["overall"]
        )

    decision_rows.append({
        "model": model_name,
        "overall_users": len(overall),
        "overall_ndcg@10": overall_ndcg,
        "overall_recall@10": overall_recall,
        "tail_users": len(tail),
        "tail_ndcg@10": float(tail["ndcg@10"].mean()),
        "tail_recall@10": float(tail["recall@10"].mean()),
        "strict_cold_users": len(cold),
        "strict_cold_ndcg@10": float(cold["ndcg@10"].mean()),
        "strict_cold_recall@10": float(cold["recall@10"].mean()),
        "warm_tail_users": len(warm),
        "warm_tail_ndcg@10": float(warm["ndcg@10"].mean()),
        "warm_tail_recall@10": float(warm["recall@10"].mean()),
        "als_overall_ndcg@10": als_overall_ndcg,
        "overall_retention_floor": retention_floor,
        "overall_guardrail_pass_point_estimate": guard_pass,
        "guardrail_margin_G": guard_point,
        "guardrail_G_ci95_low": guard_low,
        "guardrail_G_ci95_high": guard_high,
    })

decision_table = pd.DataFrame(decision_rows).sort_values(
    ["overall_guardrail_pass_point_estimate", "tail_ndcg@10"],
    ascending=[False, False],
).reset_index(drop=True)

print("CANDIDATE DECISION TABLE")
display(decision_table.round(6))

# A small config artifact for reproducibility.
run_config = {
    "notebook": "10_lightfm_hybrid.ipynb",
    "dataset_subset": NAME,
    "positive_rating_threshold": REL,
    "validation_window_start_utc": "2016-01-01T00:00:00Z",
    "tail_definition": "fewer than 20 positive train interactions",
    "strict_cold_definition": "zero positive train interactions",
    "warm_tail_definition": "1-19 positive train interactions",
    "test_split_used": False,
    "lightfm": {
        "loss": "warp",
        "components": LIGHTFM_COMPONENTS,
        "epochs": LIGHTFM_EPOCHS,
        "learning_rate": LIGHTFM_LEARNING_RATE,
        "item_alpha": LIGHTFM_ITEM_ALPHA,
        "user_alpha": LIGHTFM_USER_ALPHA,
        "max_sampled": LIGHTFM_MAX_SAMPLED,
        "num_threads": NUM_THREADS,
        "metadata_weighting": "for rows with metadata: identity feature mass 0.5, normalized metadata feature mass 0.5; empty metadata rows use identity mass 1.0",
    },
    "bootstrap_resamples": N_BOOT,
    "tie_breaking_seeds": N_TIE_SEEDS,
    "random_seed": SEED,
    "als_overall_view_a_ndcg_at_10": als_overall_ndcg,
    "retention_fraction": RETENTION_FRACTION,
    "retention_floor": retention_floor,
}

REPORT_DIR = _PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save per-model metrics with absolute user-bootstrap CIs.
absolute_rows = []
for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]
    for model_name in user_metrics:
        frame = user_metrics[model_name][scope_name].loc[users_scope]
        for metric in metrics_to_compare:
            values = frame[metric].to_numpy(dtype=np.float64)
            point, low, high = paired_bootstrap(values, draws)
            absolute_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": point,
                "ci95_low": low,
                "ci95_high": high,
            })
absolute_ci_table = pd.DataFrame(absolute_rows)

metrics_path = REPORT_DIR / "lightfm_validation_metrics.csv"
paired_path = REPORT_DIR / "lightfm_paired_validation_ci.csv"
decision_path = REPORT_DIR / "lightfm_candidate_decision.csv"
config_path = REPORT_DIR / "lightfm_run_config.json"
user_metrics_path = REPORT_DIR / "lightfm_user_metrics.csv"

absolute_ci_table.to_csv(metrics_path, index=False)
paired_ci_results.to_csv(paired_path, index=False)
decision_table.to_csv(decision_path, index=False)
(Path(config_path)).write_text(json.dumps(run_config, indent=2) + "\n", encoding="utf-8")

user_metric_export_rows = []
for model_name, scopes in user_metrics.items():
    for scope_name, frame in scopes.items():
        export = frame.reset_index().copy()
        export.insert(0, "scope", scope_name)
        export.insert(0, "model", model_name)
        user_metric_export_rows.append(export)
pd.concat(user_metric_export_rows, ignore_index=True).to_csv(user_metrics_path, index=False)

print("\nSaved reports:")
for path in (metrics_path, paired_path, decision_path, config_path, user_metrics_path):
    print(path)

print("\nOVERALL GUARDRAIL")
print(f"ALS overall View A NDCG@10: {als_overall_ndcg:.6f}")
print(f"98% retention floor:        {retention_floor:.6f}")
eligible = decision_table.loc[
    (decision_table["model"] != "ALS_only")
    & decision_table["overall_guardrail_pass_point_estimate"]
]
if len(eligible):
    selected = eligible.sort_values("tail_ndcg@10", ascending=False).iloc[0]
    print("Best LightFM candidate passing point-estimate guardrail:", selected["model"])
    print(f"Overall NDCG@10: {selected['overall_ndcg@10']:.6f} | Tail NDCG@10: {selected['tail_ndcg@10']:.6f}")
    print(
        "Guardrail G 95% CI: "
        f"[{selected['guardrail_G_ci95_low']:+.6f}, {selected['guardrail_G_ci95_high']:+.6f}]"
    )
else:
    print("No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.")

print("Test split used: No")

CANDIDATE DECISION TABLE


,model,overall_users,overall_ndcg@10,overall_recall@10,tail_users,tail_ndcg@10,tail_recall@10,strict_cold_users,strict_cold_ndcg@10,strict_cold_recall@10,warm_tail_users,warm_tail_ndcg@10,warm_tail_recall@10,als_overall_ndcg@10,overall_retention_floor,overall_guardrail_pass_point_estimate,guardrail_margin_G,guardrail_G_ci95_low,guardrail_G_ci95_high
0,ALS_only,5661,0.081462,0.106220,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,True,NaN,NaN,NaN
1,LightFM_identity,5661,0.063860,0.082599,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.015973,-0.019339,-0.012606
2,LightFM_content_v1,5661,0.062729,0.082748,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.017104,-0.020709,-0.013898
3,LightFM_TMDB_v2,5661,0.063340,0.083822,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.016492,-0.019969,-0.012963



Saved reports:
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_validation_metrics.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_paired_validation_ci.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_candidate_decision.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_run_config.json
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_user_metrics.csv

OVERALL GUARDRAIL
ALS overall View A NDCG@10: 0.081462
98% retention floor:        0.079833
No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.
Test split used: No


## 12 · How to interpret the outcome

Use `lightfm_candidate_decision.csv` for the high-level model-comparison results, `lightfm_paired_validation_ci.csv` for paired model differences, and `lightfm_validation_metrics.csv` for absolute means and confidence intervals.

Interpret the original LightFM comparison in this order:

1. **Did metadata help LightFM?** Compare `LightFM_content_v1` and `LightFM_TMDB_v2` against `LightFM_identity` on tail View A NDCG@10. A positive point estimate with an interval crossing zero is inconclusive.
2. **Did LightFM beat the existing ALS reference?** Compare overall and tail metrics plus paired intervals against `ALS_only`.
3. **Does a LightFM candidate meet the project guardrail?** A wholesale replacement must meet the 98% overall NDCG point-estimate floor under the original selection rule.
4. **What about true cold-start?** Strict-cold validation has relatively few relevant targets; treat that estimate as noisy and report hit counts and uncertainty.
5. **Selective reranking is a separate result.** Section 13 records the final test comparison for the frozen 500-rank gate, with 1,000 retained as a secondary comparison. The test results are now used for final reporting and must not be used for more tuning.

This is a controlled LightFM run, not a definitive hyperparameter search. Report where metadata helped and where it did not; do not increase metadata weight or epochs just to obtain a favourable validation result.


## Check why every model—including ALS—has zero NDCG@10, Recall@10 and HitRate@10 for the tail, warm-tail and strict-cold scopes

In [15]:
import numpy as np
import pandas as pd

# 1. Check that the existing evaluation results are available.
required_names = [
    "recommendations",
    "relevant_by_scope",
    "user_metrics",
    "eval_users",
    "tail_mask",
    "strict_cold_mask",
    "warm_tail_mask",
    "n_pos",
    "K",
    "N_TIE_SEEDS",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing notebook variables: {missing}. "
        "Do not rerun training yet; send me this message."
    )

print("=== 1. Evaluation scope audit ===")
for scope, rel_dict in relevant_by_scope.items():
    relevant_pairs = sum(len(items) for items in rel_dict.values())
    print(
        f"{scope:12s} | users={len(rel_dict):5,} "
        f"| relevant pairs={relevant_pairs:6,} "
        f"| distinct target movies="
        f"{len(set().union(*rel_dict.values())):,}"
    )

# 2. Independently recompute hit rate and NDCG from saved slates.
# Compare these values with the notebook's existing user_metrics.
discounts = 1.0 / np.log2(np.arange(2, K + 2, dtype=np.float64))
audit_rows = []

for scope, rel_dict in relevant_by_scope.items():
    users = sorted(rel_dict)

    for model_name, seeds in recommendations.items():
        per_seed_ndcg = []
        per_seed_hit_rate = []
        per_seed_hit_users = []
        per_seed_hit_pairs = []

        for seed, recs_by_user in seeds.items():
            ndcg_values = []
            hit_users = 0
            hit_pairs = 0

            for user_id in users:
                relevant = set(map(int, rel_dict[user_id]))
                slate = list(map(int, recs_by_user[user_id]))[:K]

                hits = np.array(
                    [item in relevant for item in slate],
                    dtype=np.float64,
                )

                dcg = float((hits * discounts[:len(hits)]).sum())
                ideal_length = min(len(relevant), K)
                idcg = float(discounts[:ideal_length].sum())

                ndcg_values.append(dcg / idcg if idcg > 0 else 0.0)
                hit_users += int(hits.sum() > 0)
                hit_pairs += int(hits.sum())

            per_seed_ndcg.append(float(np.mean(ndcg_values)))
            per_seed_hit_rate.append(hit_users / len(users) if users else 0.0)
            per_seed_hit_users.append(hit_users)
            per_seed_hit_pairs.append(hit_pairs)

        stored_ndcg = float(
            user_metrics[model_name][scope]["ndcg@10"].mean()
        )

        audit_rows.append({
            "scope": scope,
            "model": model_name,
            "stored_ndcg": stored_ndcg,
            "recomputed_ndcg": float(np.mean(per_seed_ndcg)),
            "mean_hit_rate": float(np.mean(per_seed_hit_rate)),
            "users_hit_per_seed": (
                f"{np.mean(per_seed_hit_users):.1f}/{len(users)}"
            ),
            "hit_pairs_per_seed": float(np.mean(per_seed_hit_pairs)),
        })

audit = pd.DataFrame(audit_rows)

print("\n=== 2. Independent metric check ===")
display(
    audit.round(6).sort_values(["scope", "model"]).reset_index(drop=True)
)

# 3. Check what kinds of movies actually enter the top-10 recommendations.
# This separates "not recommending tail movies" from "recommending
# tail movies, but not the correct held-out tail targets".
exposure_rows = []

for model_name, seeds in recommendations.items():
    all_items = []

    for seed, recs_by_user in seeds.items():
        for user_id in eval_users:
            all_items.extend(
                map(int, recs_by_user[int(user_id)][:K])
            )

    items = np.asarray(all_items, dtype=np.int64)

    exposure_rows.append({
        "model": model_name,
        "recommendation_slots": len(items),
        "tail_item_exposure_pct": 100 * float(tail_mask[items].mean()),
        "warm_tail_exposure_pct": 100 * float(warm_tail_mask[items].mean()),
        "strict_cold_exposure_pct": 100 * float(strict_cold_mask[items].mean()),
        "head_item_exposure_pct": 100 * float((n_pos[items] >= 20).mean()),
        "unique_items_recommended": int(np.unique(items).size),
    })

print("\n=== 3. Recommendation catalogue exposure ===")
display(
    pd.DataFrame(exposure_rows)
    .round(3)
    .sort_values("model")
    .reset_index(drop=True)
)

# 4. Explicitly check the ALS factor-availability constraint.
if "item_has_als_factors" in globals():
    print("\n=== 4. ALS item-factor availability ===")
    for label, mask in [
        ("All catalogue items", np.ones_like(n_pos, dtype=bool)),
        ("Tail items (<20)", tail_mask),
        ("Warm-tail items (1-19)", warm_tail_mask),
        ("Strict-cold items (0)", strict_cold_mask),
    ]:
        total = int(mask.sum())
        available = int((mask & item_has_als_factors).sum())
        print(
            f"{label:28s}: {available:,}/{total:,} "
            f"have learned ALS factors"
        )

print("\nDiagnostic complete. No models were retrained.")

=== 1. Evaluation scope audit ===
overall      | users=5,661 | relevant pairs=25,406 | distinct target movies=4,621
tail         | users=  948 | relevant pairs= 1,585 | distinct target movies=1,266
strict_cold  | users=   50 | relevant pairs=    52 | distinct target movies=47
warm_tail    | users=  931 | relevant pairs= 1,533 | distinct target movies=1,219

=== 2. Independent metric check ===


,scope,model,stored_ndcg,recomputed_ndcg,mean_hit_rate,users_hit_per_seed,hit_pairs_per_seed
0,overall,ALS_only,0.081462,0.081462,0.302067,1710.0/5661,2403.0
1,overall,LightFM_TMDB_v2,0.063340,0.063340,0.254902,1443.0/5661,1928.0
2,overall,LightFM_content_v1,0.062729,0.062729,0.251546,1424.0/5661,1912.0
3,overall,LightFM_identity,0.063860,0.063860,0.255255,1445.0/5661,1936.0
4,strict_cold,ALS_only,0.000000,0.000000,0.000000,0.0/50,0.0
5,strict_cold,LightFM_TMDB_v2,0.000000,0.000000,0.000000,0.0/50,0.0
6,strict_cold,LightFM_content_v1,0.000000,0.000000,0.000000,0.0/50,0.0
7,strict_cold,LightFM_identity,0.000000,0.000000,0.000000,0.0/50,0.0
8,tail,ALS_only,0.000000,0.000000,0.000000,0.0/948,0.0
9,tail,LightFM_TMDB_v2,0.000000,0.000000,0.000000,0.0/948,0.0



=== 3. Recommendation catalogue exposure ===


,model,recommendation_slots,tail_item_exposure_pct,warm_tail_exposure_pct,strict_cold_exposure_pct,head_item_exposure_pct,unique_items_recommended
0,ALS_only,566100,0.000,0.000,0.000,100.000,1202
1,LightFM_TMDB_v2,566100,0.493,0.456,0.037,99.507,1173
2,LightFM_content_v1,566100,0.094,0.092,0.002,99.906,1246
3,LightFM_identity,566100,0.004,0.004,0.000,99.996,1313



=== 4. ALS item-factor availability ===
All catalogue items         : 16,146/17,241 have learned ALS factors
Tail items (<20)            : 10,502/11,597 have learned ALS factors
Warm-tail items (1-19)      : 10,502/10,502 have learned ALS factors
Strict-cold items (0)       : 0/1,095 have learned ALS factors

Diagnostic complete. No models were retrained.


In [16]:
# Diagnostic A: item-index alignment and bounds
import numpy as np

assert np.array_equal(
    item_map["i"].to_numpy(),
    np.arange(n_items)
), "item_map indices are not contiguous"

assert len(n_pos) == n_items
assert len(tail_mask) == n_items
assert len(warm_tail_mask) == n_items
assert len(strict_cold_mask) == n_items
assert Q_als.shape[0] == n_items

assert X_v1.shape[0] == n_items
assert X_tmdb_v2.shape[0] == n_items

assert np.array_equal(tail_mask, n_pos < 20)
assert np.array_equal(warm_tail_mask, (n_pos >= 1) & (n_pos < 20))
assert np.array_equal(strict_cold_mask, n_pos == 0)

assert np.all(~(warm_tail_mask & strict_cold_mask))
assert np.array_equal(tail_mask, warm_tail_mask | strict_cold_mask)

# Every recommendation must be a valid internal catalogue index.
for model_name, seeds in recommendations.items():
    for seed, recs_by_user in seeds.items():
        for user_id, slate in recs_by_user.items():
            slate = np.asarray(slate, dtype=np.int64)
            assert np.all((slate >= 0) & (slate < n_items)), (
                f"Out-of-range item index: {model_name}, seed={seed}, user={user_id}"
            )

print("PASS: item-index bounds, masks, feature rows and recommendation IDs align.")
print("Catalogue size:", n_items)
print("Tail:", int(tail_mask.sum()))
print("Warm tail:", int(warm_tail_mask.sum()))
print("Strict cold:", int(strict_cold_mask.sum()))

PASS: item-index bounds, masks, feature rows and recommendation IDs align.
Catalogue size: 17241
Tail: 11597
Warm tail: 10502
Strict cold: 1095


In [17]:
# Diagnostic A: item-index alignment and bounds
import numpy as np

assert np.array_equal(
    item_map["i"].to_numpy(),
    np.arange(n_items)
), "item_map indices are not contiguous"

assert len(n_pos) == n_items
assert len(tail_mask) == n_items
assert len(warm_tail_mask) == n_items
assert len(strict_cold_mask) == n_items
assert Q_als.shape[0] == n_items

assert X_v1.shape[0] == n_items
assert X_tmdb_v2.shape[0] == n_items

assert np.array_equal(tail_mask, n_pos < 20)
assert np.array_equal(warm_tail_mask, (n_pos >= 1) & (n_pos < 20))
assert np.array_equal(strict_cold_mask, n_pos == 0)

assert np.all(~(warm_tail_mask & strict_cold_mask))
assert np.array_equal(tail_mask, warm_tail_mask | strict_cold_mask)

# Every recommendation must be a valid internal catalogue index.
for model_name, seeds in recommendations.items():
    for seed, recs_by_user in seeds.items():
        for user_id, slate in recs_by_user.items():
            slate = np.asarray(slate, dtype=np.int64)
            assert np.all((slate >= 0) & (slate < n_items)), (
                f"Out-of-range item index: {model_name}, seed={seed}, user={user_id}"
            )

print("PASS: item-index bounds, masks, feature rows and recommendation IDs align.")
print("Catalogue size:", n_items)
print("Tail:", int(tail_mask.sum()))
print("Warm tail:", int(warm_tail_mask.sum()))
print("Strict cold:", int(strict_cold_mask.sum()))

PASS: item-index bounds, masks, feature rows and recommendation IDs align.
Catalogue size: 17241
Tail: 11597
Warm tail: 10502
Strict cold: 1095


## Diagnostic B: full catalogue vs restricted candidate pools

In [18]:
# Reuses fitted models, score_functions, recommendations and evaluation targets.

import numpy as np
import pandas as pd

K_DIAG = K
discounts_diag = 1.0 / np.log2(
    np.arange(2, K_DIAG + 2, dtype=np.float64)
)

def evaluate_restricted_pool(model_name, scope_name, candidate_mask):
    """Rank a restricted candidate pool and average metrics over tie seeds."""
    rel_dict = relevant_by_scope[scope_name]
    users = np.asarray(sorted(rel_dict), dtype=np.int64)
    eligible_catalogue = np.asarray(candidate_mask, dtype=bool)

    per_seed_results = []

    for tie_seed in range(N_TIE_SEEDS):
        rows = []

        for start in range(0, len(users), SCORE_BATCH_SIZE):
            user_batch = users[start:start + SCORE_BATCH_SIZE]
            batch_scores = score_functions[model_name](user_batch)

            for row_idx, user_id in enumerate(user_batch):
                scores = batch_scores[row_idx].astype(np.float64, copy=True)

                # Candidate pool restriction and seen-item filtering.
                eligible = eligible_catalogue.copy()
                eligible[seen_train[int(user_id)].indices] = False

                # Standard ALS has no learned factors for strict-cold items.
                if model_name == "ALS_only":
                    eligible &= item_has_als_factors

                candidate_ids = np.flatnonzero(eligible)
                if len(candidate_ids) < K_DIAG:
                    raise RuntimeError(
                        f"Only {len(candidate_ids)} eligible candidates for "
                        f"{model_name}, user {user_id}"
                    )

                candidate_scores = scores[candidate_ids]

                # Match the existing seeded tie-breaking convention.
                rng = np.random.default_rng(
                    np.random.SeedSequence(
                        [SEED, int(tie_seed), int(user_id)]
                    )
                )
                noise = rng.random(len(candidate_ids))
                scale = max(1.0, float(np.max(np.abs(candidate_scores))))
                adjusted = candidate_scores + noise * (1e-8 * scale)

                top_idx = np.argpartition(adjusted, -K_DIAG)[-K_DIAG:]
                top_idx = top_idx[
                    np.argsort(adjusted[top_idx])[::-1]
                ]
                slate = candidate_ids[top_idx]

                relevant = set(map(int, rel_dict[int(user_id)]))
                hits = np.asarray(
                    [int(item) in relevant for item in slate],
                    dtype=np.float64,
                )

                idcg_len = min(len(relevant), K_DIAG)
                idcg = float(discounts_diag[:idcg_len].sum())
                ndcg = (
                    float((hits * discounts_diag).sum() / idcg)
                    if idcg > 0 else 0.0
                )
                recall = (
                    float(hits.sum() / len(relevant))
                    if relevant else 0.0
                )

                rows.append({
                    "ndcg@10": ndcg,
                    "recall@10": recall,
                    "hit_rate@10": float(hits.sum() > 0),
                    "hit_count@10": float(hits.sum()),
                })

        per_seed_results.append(pd.DataFrame(rows))

    averaged = sum(per_seed_results) / len(per_seed_results)
    return {
        metric: float(averaged[metric].mean())
        for metric in ["ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"]
    }


result_rows = []

# A. Existing full-catalogue results, calculated from the saved top-10 lists.
for scope_name in ["tail", "warm_tail", "strict_cold"]:
    for model_name in recommendations:
        frame = user_metrics[model_name][scope_name]
        result_rows.append({
            "pool": "full_catalogue_existing",
            "scope": scope_name,
            "model": model_name,
            "users": len(frame),
            "ndcg@10": float(frame["ndcg@10"].mean()),
            "recall@10": float(frame["recall@10"].mean()),
            "hit_rate@10": float(frame["hit_rate@10"].mean()),
            "hit_count@10": float(frame["hit_count@10"].mean()),
        })

# B. Common warm-tail pool: all four models compete for the same items.
for model_name in recommendations:
    for scope_name in ["warm_tail"]:
        metrics = evaluate_restricted_pool(
            model_name, scope_name, warm_tail_mask
        )
        result_rows.append({
            "pool": "warm_tail_only_common",
            "scope": scope_name,
            "model": model_name,
            "users": len(relevant_by_scope[scope_name]),
            **metrics,
        })

# C. Tail pool: ALS can score learned-factor warm-tail items only;
# LightFM can score both warm-tail and strict-cold items.
for model_name in recommendations:
    pool_mask = warm_tail_mask if model_name == "ALS_only" else tail_mask
    for scope_name in ["tail", "warm_tail"]:
        metrics = evaluate_restricted_pool(
            model_name, scope_name, pool_mask
        )
        result_rows.append({
            "pool": (
                "tail_only_ALS_known_items"
                if model_name == "ALS_only"
                else "tail_only_full_tail"
            ),
            "scope": scope_name,
            "model": model_name,
            "users": len(relevant_by_scope[scope_name]),
            **metrics,
        })

ranking_comparison = pd.DataFrame(result_rows)

print("=== Restricted-pool ranking comparison ===")
display(
    ranking_comparison.sort_values(
        ["scope", "pool", "model"]
    ).round(6).reset_index(drop=True)
)

print("\n=== Tail-only exposure in existing full-catalogue top-10 lists ===")
display(
    pd.DataFrame([
        {
            "model": model_name,
            "tail_exposure_pct": 100 * np.mean([
                tail_mask[int(item)]
                for seed, users in seeds.items()
                for user_id, slate in users.items()
                for item in slate[:K_DIAG]
            ]),
            "warm_tail_exposure_pct": 100 * np.mean([
                warm_tail_mask[int(item)]
                for seed, users in seeds.items()
                for user_id, slate in users.items()
                for item in slate[:K_DIAG]
            ]),
        }
        for model_name, seeds in recommendations.items()
    ]).round(4))


=== Restricted-pool ranking comparison ===


,pool,scope,model,users,ndcg@10,recall@10,hit_rate@10,hit_count@10
0,full_catalogue_existing,strict_cold,ALS_only,50,0.000000,0.000000,0.000000,0.000000
1,full_catalogue_existing,strict_cold,LightFM_TMDB_v2,50,0.000000,0.000000,0.000000,0.000000
2,full_catalogue_existing,strict_cold,LightFM_content_v1,50,0.000000,0.000000,0.000000,0.000000
3,full_catalogue_existing,strict_cold,LightFM_identity,50,0.000000,0.000000,0.000000,0.000000
4,full_catalogue_existing,tail,ALS_only,948,0.000000,0.000000,0.000000,0.000000
5,full_catalogue_existing,tail,LightFM_TMDB_v2,948,0.000000,0.000000,0.000000,0.000000
6,full_catalogue_existing,tail,LightFM_content_v1,948,0.000000,0.000000,0.000000,0.000000
7,full_catalogue_existing,tail,LightFM_identity,948,0.000000,0.000000,0.000000,0.000000
8,tail_only_ALS_known_items,tail,ALS_only,948,0.033015,0.057517,0.081224,0.088608
9,tail_only_full_tail,tail,LightFM_TMDB_v2,948,0.028473,0.052067,0.077004,0.082278



=== Tail-only exposure in existing full-catalogue top-10 lists ===


,model,tail_exposure_pct,warm_tail_exposure_pct
0,ALS_only,0.0000,0.0000
1,LightFM_identity,0.0035,0.0035
2,LightFM_content_v1,0.0936,0.0919
3,LightFM_TMDB_v2,0.4928,0.4557


In [19]:

import numpy as np
import pandas as pd

# Evaluate only warm-tail targets so every model uses a comparable target set.
warm_tail_relevant = relevant_by_scope["warm_tail"]

# Map model names to readable labels.
model_names = {
    "ALS_only": "ALS",
    "LightFM_identity": "LightFM Identity",
    "LightFM_content_v1": "LightFM Content-v1",
    "LightFM_TMDB_v2": "LightFM TMDB-v2",
}

# Check the data structures before scoring.
print("Warm-tail users:", len(warm_tail_relevant))
print("Warm-tail relevant pairs:", sum(len(items) for items in warm_tail_relevant.values()))
print("Models:", list(score_functions.keys()))

assert all(name in score_functions for name in model_names), (
    "A model name differs from the keys in score_functions. "
    "Print score_functions.keys() and update model_names."
)

# Calculate the rank of each relevant movie among all eligible catalogue items.
# Rank 1 = highest score. Seen training items are excluded.
rank_rows = []

for model_key, model_label in model_names.items():
    score_fn = score_functions[model_key]

    for user_id, target_items in warm_tail_relevant.items():
        target_items = np.asarray(list(target_items), dtype=int)
        if len(target_items) == 0:
            continue

        # Score the full catalogue for this user.
        scores = np.asarray(score_fn([user_id]))[0].astype(float, copy=True)

        # Apply the same basic eligibility rules as recommendation generation.
        seen_items = seen_train[user_id].indices
        scores[seen_items] = -np.inf

        # ALS cannot recommend items without learned ALS item factors.
        if model_key == "ALS_only":
            scores[~item_has_als_factors] = -np.inf

        # Exclude targets that aren't eligible or have invalid scores.
        valid_targets = target_items[
            (target_items >= 0)
            & (target_items < len(scores))
            & np.isfinite(scores[target_items])
        ]

        # Raw-score ranks; ties are unlikely but are not tie-break-seed averaged.
        for item_id in valid_targets:
            target_score = scores[item_id]
            rank = 1 + int(np.sum(scores > target_score))

            rank_rows.append({
                "model": model_label,
                "user_id": user_id,
                "item_id": int(item_id),
                "rank": rank,
            })

rank_df = pd.DataFrame(rank_rows)

print("Ranked target pairs:", len(rank_df))
display(rank_df.head())


Warm-tail users: 931
Warm-tail relevant pairs: 1533
Models: ['ALS_only', 'LightFM_identity', 'LightFM_content_v1', 'LightFM_TMDB_v2']
Ranked target pairs: 6132


,model,user_id,item_id,rank
0,ALS,100,13211,1834
1,ALS,105,8550,1003
2,ALS,127,11487,2210
3,ALS,127,11323,7844
4,ALS,127,11335,9330


In [20]:

# Pair-level: how high do relevant warm-tail movies rank?
rank_summary = (
    rank_df.groupby("model")["rank"]
    .agg(
        target_pairs="count",
        median_rank="median",
        mean_rank="mean",
        p90_rank=lambda x: x.quantile(0.90),
    )
)

for k in [10, 50, 100, 500]:
    rank_summary[f"targets_in_top_{k}_pct"] = (
        rank_df.assign(in_top_k=rank_df["rank"] <= k)
        .groupby("model")["in_top_k"]
        .mean()
        .mul(100)
    )

# User-level: percentage of users with at least one relevant warm-tail
# movie in the top k. This is closer to a hit-rate interpretation.
user_hit_rows = []

for (model, user_id), group in rank_df.groupby(["model", "user_id"]):
    row = {"model": model, "user_id": user_id}
    for k in [10, 50, 100, 500]:
        row[f"hit_{k}"] = bool((group["rank"] <= k).any())
    user_hit_rows.append(row)

user_hits_df = pd.DataFrame(user_hit_rows)

for k in [10, 50, 100, 500]:
    hit_pct = (
        user_hits_df.groupby("model")[f"hit_{k}"]
        .mean()
        .mul(100)
    )
    rank_summary[f"users_with_hit_top_{k}_pct"] = hit_pct

display(rank_summary.round(4))


,target_pairs,median_rank,mean_rank,p90_rank,targets_in_top_10_pct,targets_in_top_50_pct,targets_in_top_100_pct,targets_in_top_500_pct,users_with_hit_top_10_pct,users_with_hit_top_50_pct,users_with_hit_top_100_pct,users_with_hit_top_500_pct
model,,,,,,,,,,,,
ALS,1533,2980.0,3972.9830,8143.0,0.0,0.0000,0.0000,0.2609,0.0,0.0000,0.0000,0.4296
LightFM Content-v1,1533,2478.0,3333.7632,7343.8,0.0,0.1957,0.5871,6.6536,0.0,0.1074,0.6445,9.0226
LightFM Identity,1533,2721.0,3685.9152,8010.2,0.0,0.0000,0.3262,3.9139,0.0,0.0000,0.5371,5.2632
LightFM TMDB-v2,1533,2571.0,3391.8637,6926.2,0.0,0.0652,0.3262,4.6314,0.0,0.1074,0.4296,6.2299


In [21]:
print("Relevant scopes:", list(relevant_by_scope.keys()))
print("\nRecommendation object:")
print(type(recommendations))

if isinstance(recommendations, dict):
    print("Keys:", list(recommendations.keys())[:10])
elif isinstance(recommendations, pd.DataFrame):
    print("Columns:", recommendations.columns.tolist())
    print(recommendations.head(3))

print("\nUser metrics:")
if isinstance(user_metrics, pd.DataFrame):
    print("Columns:", user_metrics.columns.tolist())
    print(user_metrics.head(3))
else:
    print(type(user_metrics))

Relevant scopes: ['overall', 'tail', 'strict_cold', 'warm_tail']

Recommendation object:
<class 'dict'>
Keys: ['ALS_only', 'LightFM_identity', 'LightFM_content_v1', 'LightFM_TMDB_v2']

User metrics:
<class 'dict'>


In [22]:

for model in recommendations:
    print(f"\n{'=' * 60}")
    print("MODEL:", model)

    rec = recommendations[model]
    print("Recommendation type:", type(rec))

    if isinstance(rec, dict):
        print("Recommendation keys:", list(rec.keys())[:15])
        for key, value in list(rec.items())[:3]:
            print(f"  {key}: type={type(value)}, shape={getattr(value, 'shape', None)}")
    else:
        print("Shape:", getattr(rec, "shape", None))
        print("Sample:", repr(rec)[:400])

    metrics = user_metrics.get(model)
    print("Metrics type:", type(metrics))

    if isinstance(metrics, dict):
        print("Metric keys:", list(metrics.keys())[:15])
        for key, value in list(metrics.items())[:3]:
            print(f"  {key}: type={type(value)}, shape={getattr(value, 'shape', None)}")
    elif metrics is not None:
        print("Metrics shape:", getattr(metrics, "shape", None))
        print("Metric sample:", repr(metrics)[:400])



MODEL: ALS_only
Recommendation type: <class 'dict'>
Recommendation keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  0: type=<class 'dict'>, shape=None
  1: type=<class 'dict'>, shape=None
  2: type=<class 'dict'>, shape=None
Metrics type: <class 'dict'>
Metric keys: ['overall', 'tail', 'strict_cold', 'warm_tail']
  overall: type=<class 'pandas.core.frame.DataFrame'>, shape=(5661, 4)
  tail: type=<class 'pandas.core.frame.DataFrame'>, shape=(948, 4)
  strict_cold: type=<class 'pandas.core.frame.DataFrame'>, shape=(50, 4)

MODEL: LightFM_identity
Recommendation type: <class 'dict'>
Recommendation keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  0: type=<class 'dict'>, shape=None
  1: type=<class 'dict'>, shape=None
  2: type=<class 'dict'>, shape=None
Metrics type: <class 'dict'>
Metric keys: ['overall', 'tail', 'strict_cold', 'warm_tail']
  overall: type=<class 'pandas.core.frame.DataFrame'>, shape=(5661, 4)
  tail: type=<class 'pandas.core.frame.DataFrame'>, shape=(948, 4)
  strict_cold: type=<class 'pan

In [23]:

sample_model = "ALS_only"

for seed in [0]:
    sample = recommendations[sample_model][seed]
    print("Seed record type:", type(sample))
    print("Number of entries:", len(sample))

    if isinstance(sample, dict):
        first_user = next(iter(sample))
        print("First user ID:", first_user)
        print("User record type:", type(sample[first_user]))
        print("User record:", repr(sample[first_user])[:1000])
    else:
        print("Sample:", repr(sample)[:1000])

print("\nPer-user metric columns:")
print(user_metrics["ALS_only"]["overall"].columns.tolist())
print(user_metrics["ALS_only"]["warm_tail"].head(3))


Seed record type: <class 'dict'>
Number of entries: 5661
First user ID: 4
User record type: <class 'numpy.ndarray'>
User record: array([ 2320,   305, 11629, 10244, 12488,  3207,  8330, 12908,  2491,
       12751], dtype=int32)

Per-user metric columns:
['ndcg@10', 'recall@10', 'hit_rate@10', 'hit_count@10']
     ndcg@10  recall@10  hit_rate@10  hit_count@10
u                                                 
100      0.0        0.0          0.0           0.0
105      0.0        0.0          0.0           0.0
127      0.0        0.0          0.0           0.0


In [24]:

print("Warm-tail mask:", type(warm_tail_mask), getattr(warm_tail_mask, "shape", None))
print("Relevant warm-tail:", type(relevant_by_scope["warm_tail"]))

# Inspect a user's relevant warm-tail targets.
sample_user = next(iter(relevant_by_scope["warm_tail"]))
print("\nSample user:", sample_user)
print("Relevant warm-tail targets:",
      relevant_by_scope["warm_tail"][sample_user])

# Inspect catalogue metadata and popularity information.
print("\nitem_map type:", type(item_map))
print("item_map columns:", getattr(item_map, "columns", None))
if isinstance(item_map, pd.DataFrame):
    print(item_map.head(3).to_string())

print("\nn_pos type/shape:", type(n_pos), getattr(n_pos, "shape", None))
print("Warm-tail item count:", int(np.asarray(warm_tail_mask).sum()))

# Check how user IDs are represented in the validation relevance map.
print("\nRecommendation user ID example:",
      next(iter(recommendations["ALS_only"][0])))


Warm-tail mask: <class 'numpy.ndarray'> (17241,)
Relevant warm-tail: <class 'dict'>

Sample user: 100
Relevant warm-tail targets: {13211}

item_map type: <class 'pandas.core.frame.DataFrame'>
item_map columns: Index(['i', 'movieId'], dtype='object')
   i  movieId
0  0        1
1  1        2
2  2        3

n_pos type/shape: <class 'numpy.ndarray'> (17241,)
Warm-tail item count: 10502

Recommendation user ID example: 4


In [25]:

import inspect

print("=== top_k_from_scores ===")
print(inspect.getsource(top_k_from_scores))

print("\n=== Scoring function signatures ===")
for name, fn in score_functions.items():
    print(name, inspect.signature(fn))

print("\n=== Relevant ALS variables ===")
for name in [
    "item_has_als_factors",
    "seen_train",
    "u_has_pos",
    "popularity_scores",
]:
    if name in globals():
        value = globals()[name]
        print(
            name,
            type(value),
            getattr(value, "shape", None)
        )
    else:
        print(name, "not found")


=== top_k_from_scores ===
def top_k_from_scores(scores, user_id, model_name, tie_seed, k=K):
    row = np.asarray(scores, dtype=np.float64).copy()

    # Users with no positive train interactions receive the agreed fallback list.
    if not u_has_pos[user_id]:
        row = popularity_fallback_scores(user_id)
    elif model_name == "ALS_only":
        row[~item_has_als_factors] = -np.inf

    # Never recommend any movie that appeared in the user's training history.
    row[seen_train[user_id].indices] = -np.inf
    eligible = np.isfinite(row)
    if int(eligible.sum()) < k:
        raise RuntimeError(f"Only {int(eligible.sum())} eligible movies for user {user_id}")

    # Seeded per-user tie-break. The same noise vector is used across models for a given
    # user/seed; epsilon is tiny relative to the model score scale.
    tie_rng = np.random.default_rng(np.random.SeedSequence([SEED, int(tie_seed), int(user_id)]))
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.m

In [26]:

# Controlled ALS + Content-v1 warm-tail reranking experiment
# No retraining. Keeps the original recommendations untouched.

import numpy as np
import pandas as pd

BASE_MODEL = "ALS_only"
TAIL_MODEL = "LightFM_content_v1"
K = 10
RESERVED_SLOTS = [0, 1, 2, 3]

warm_tail_items = np.flatnonzero(warm_tail_mask)
warm_tail_relevant = relevant_by_scope["warm_tail"]
overall_relevant = relevant_by_scope["overall"]

# Evaluate the same users across all settings.
eval_users = sorted(overall_relevant.keys())
assert len(eval_users) == 5661, f"Unexpected evaluation user count: {len(eval_users)}"

def ranking_metrics_at_10(recs, relevant):
    """Return per-user NDCG@10, Recall@10, HitRate@10 and hit count."""
    rows = []

    for user_id in eval_users:
        rec = np.asarray(recs[user_id], dtype=int)
        truth = set(relevant.get(user_id, set()))

        hits = np.array([int(item in truth) for item in rec], dtype=float)
        discounts = 1.0 / np.log2(np.arange(2, K + 2))
        dcg = float(np.sum(hits * discounts))

        ideal_hits = min(len(truth), K)
        idcg = float(np.sum(discounts[:ideal_hits])) if ideal_hits else 0.0

        rows.append({
            "user_id": user_id,
            "ndcg@10": dcg / idcg if idcg else 0.0,
            "recall@10": hits.sum() / len(truth) if truth else 0.0,
            "hit_rate@10": float(hits.sum() > 0),
            "hit_count@10": float(hits.sum()),
        })

    return pd.DataFrame(rows).set_index("user_id")


# Precompute candidate lists per user and seed.
# The reranker uses the content model's full-catalogue scores, filtered to
# eligible warm-tail movies. Seeded tie-breaking matches the existing protocol.
rerank_rows = []
rerank_user_metrics = {}

for seed in range(10):
    base_seed_recs = recommendations[BASE_MODEL][seed]
    new_seed_recs = {k: {} for k in RESERVED_SLOTS}

    for user_id in eval_users:
        base_list = list(map(int, base_seed_recs[user_id]))

        # Preserve the exact original baseline at k=0.
        new_seed_recs[0][user_id] = np.asarray(base_list, dtype=np.int32)

        if not u_has_pos[user_id]:
            # For fallback users, preserve the established popularity fallback.
            continue

        content_scores = np.asarray(
            score_functions[TAIL_MODEL]([user_id])
        )[0].astype(np.float64, copy=True)

        # Match candidate eligibility rules.
        content_scores[seen_train[user_id].indices] = -np.inf
        content_scores[~np.asarray(warm_tail_mask, dtype=bool)] = -np.inf
        content_scores[~item_has_als_factors] = -np.inf

        eligible = np.isfinite(content_scores)
        if not eligible.any():
            for slot_count in RESERVED_SLOTS[1:]:
                new_seed_recs[slot_count][user_id] = np.asarray(
                    base_list, dtype=np.int32
                )
            continue

        # Match the existing deterministic per-user/seed tie-break.
        tie_rng = np.random.default_rng(
            np.random.SeedSequence([SEED, int(seed), int(user_id)])
        )
        tie_noise = tie_rng.random(n_items)
        scale = max(1.0, float(np.max(np.abs(content_scores[eligible]))))
        adjusted = content_scores.copy()
        adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

        candidate_order = np.argsort(adjusted)[::-1]
        tail_candidates = [
            int(item) for item in candidate_order
            if eligible[item]
        ]

        for slot_count in RESERVED_SLOTS[1:]:
            # Take the best tail candidates, excluding movies already in ALS top 10.
            selected = []
            base_set = set(base_list)

            for item in tail_candidates:
                if item not in base_set:
                    selected.append(item)
                if len(selected) == slot_count:
                    break

            # If insufficient new tail candidates exist, keep the base list.
            if len(selected) < slot_count:
                new_seed_recs[slot_count][user_id] = np.asarray(
                    base_list, dtype=np.int32
                )
                continue

            # Remove the lowest-ranked ALS items, then insert selected tail items.
            kept = base_list[:K - slot_count]
            new_seed_recs[slot_count][user_id] = np.asarray(
                kept + selected, dtype=np.int32
            )

    # Evaluate each reservation setting for this seed.
    for slot_count in RESERVED_SLOTS:
        recs = new_seed_recs[slot_count]

        overall_metrics = ranking_metrics_at_10(recs, overall_relevant)
        tail_metrics = ranking_metrics_at_10(recs, warm_tail_relevant)

        for user_id in eval_users:
            rerank_rows.append({
                "seed": seed,
                "reserved_slots": slot_count,
                "user_id": user_id,
                "overall_ndcg@10": overall_metrics.loc[user_id, "ndcg@10"],
                "overall_recall@10": overall_metrics.loc[user_id, "recall@10"],
                "overall_hit_rate@10": overall_metrics.loc[user_id, "hit_rate@10"],
                "warm_tail_ndcg@10": tail_metrics.loc[user_id, "ndcg@10"],
                "warm_tail_recall@10": tail_metrics.loc[user_id, "recall@10"],
                "warm_tail_hit_rate@10": tail_metrics.loc[user_id, "hit_rate@10"],
            })

rerank_results = pd.DataFrame(rerank_rows)

summary = (
    rerank_results
    .groupby("reserved_slots")
    .agg(
        overall_ndcg=("overall_ndcg@10", "mean"),
        overall_recall=("overall_recall@10", "mean"),
        overall_hit_rate=("overall_hit_rate@10", "mean"),
        warm_tail_ndcg=("warm_tail_ndcg@10", "mean"),
        warm_tail_recall=("warm_tail_recall@10", "mean"),
        warm_tail_hit_rate=("warm_tail_hit_rate@10", "mean"),
    )
)

baseline_ndcg = summary.loc[0, "overall_ndcg"]
summary["overall_ndcg_loss_vs_baseline"] = baseline_ndcg - summary["overall_ndcg"]
summary["overall_ndcg_loss_pct"] = (
    summary["overall_ndcg_loss_vs_baseline"] / baseline_ndcg * 100
)

display(summary.round(6))


,overall_ndcg,overall_recall,overall_hit_rate,warm_tail_ndcg,warm_tail_recall,warm_tail_hit_rate,overall_ndcg_loss_vs_baseline,overall_ndcg_loss_pct
reserved_slots,,,,,,,,
0,0.081462,0.106220,0.302067,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.078486,0.099389,0.289172,0.000466,0.001438,0.002296,0.002976,3.653426
2,0.075060,0.091794,0.272920,0.000979,0.002946,0.004769,0.006403,7.859582
3,0.071285,0.083695,0.255079,0.001398,0.004123,0.006536,0.010177,12.493137


In [27]:
# Overall metrics: all 5,661 evaluation users.
overall_summary = (
    rerank_results.groupby("reserved_slots")
    .agg(
        overall_ndcg=("overall_ndcg@10", "mean"),
        overall_recall=("overall_recall@10", "mean"),
        overall_hit_rate=("overall_hit_rate@10", "mean"),
    )
)

# Warm-tail metrics: only users in the established warm-tail evaluation scope.
warm_tail_users = set(relevant_by_scope["warm_tail"].keys())
warm_results = rerank_results[
    rerank_results["user_id"].isin(warm_tail_users)
]

warm_summary = (
    warm_results.groupby("reserved_slots")
    .agg(
        warm_tail_users=("user_id", "nunique"),
        warm_tail_ndcg=("warm_tail_ndcg@10", "mean"),
        warm_tail_recall=("warm_tail_recall@10", "mean"),
        warm_tail_hit_rate=("warm_tail_hit_rate@10", "mean"),
    )
)

corrected_summary = overall_summary.join(warm_summary)

baseline_ndcg = corrected_summary.loc[0, "overall_ndcg"]
corrected_summary["overall_ndcg_loss"] = (
    baseline_ndcg - corrected_summary["overall_ndcg"]
)
corrected_summary["overall_ndcg_loss_pct"] = (
    corrected_summary["overall_ndcg_loss"] / baseline_ndcg * 100
)

baseline_tail_recall = corrected_summary.loc[0, "warm_tail_recall"]
baseline_tail_hit = corrected_summary.loc[0, "warm_tail_hit_rate"]

corrected_summary["tail_recall_gain"] = (
    corrected_summary["warm_tail_recall"] - baseline_tail_recall
)
corrected_summary["tail_hit_rate_gain"] = (
    corrected_summary["warm_tail_hit_rate"] - baseline_tail_hit
)

print("Warm-tail evaluation users:", len(warm_tail_users))
display(corrected_summary.round(6))


Warm-tail evaluation users: 931


,overall_ndcg,overall_recall,overall_hit_rate,warm_tail_users,warm_tail_ndcg,warm_tail_recall,warm_tail_hit_rate,overall_ndcg_loss,overall_ndcg_loss_pct,tail_recall_gain,tail_hit_rate_gain
reserved_slots,,,,,,,,,,,
0,0.081462,0.106220,0.302067,931,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.078486,0.099389,0.289172,931,0.002836,0.008746,0.013963,0.002976,3.653426,0.008746,0.013963
2,0.075060,0.091794,0.272920,931,0.005952,0.017912,0.029001,0.006403,7.859582,0.017912,0.029001
3,0.071285,0.083695,0.255079,931,0.008498,0.025073,0.039742,0.010177,12.493137,0.025073,0.039742


In [28]:

# Paired user-level bootstrap confidence intervals.
# 95% percentile CIs for changes versus k=0.
# No scoring or retraining.

rng = np.random.default_rng(2026)
N_BOOT = 3000

# Average across the 10 tie-break seeds first.
metric_cols = [
    "overall_ndcg@10",
    "warm_tail_recall@10",
    "warm_tail_hit_rate@10",
]

user_level = (
    rerank_results
    .groupby(["reserved_slots", "user_id"])[metric_cols]
    .mean()
)

warm_users = sorted(relevant_by_scope["warm_tail"].keys())
baseline_all = user_level.loc[0]

def bootstrap_mean_ci(differences, n_boot=N_BOOT, batch_size=200):
    """Bootstrap the mean paired difference across users."""
    values = np.asarray(differences, dtype=float)
    n = len(values)
    boot_means = np.empty(n_boot, dtype=float)

    for start in range(0, n_boot, batch_size):
        batch_n = min(batch_size, n_boot - start)
        idx = rng.integers(0, n, size=(batch_n, n))
        boot_means[start:start + batch_n] = values[idx].mean(axis=1)

    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    return float(values.mean()), float(lo), float(hi)

ci_rows = []

for k in [1, 2, 3]:
    current = user_level.loc[k]

    # Overall NDCG: all evaluation users.
    paired_ndcg = (
        current.loc[baseline_all.index, "overall_ndcg@10"].to_numpy()
        - baseline_all["overall_ndcg@10"].to_numpy()
    )

    mean, lo, hi = bootstrap_mean_ci(paired_ndcg)
    ci_rows.append({
        "reserved_slots": k,
        "metric": "Overall NDCG@10 change",
        "mean_change": mean,
        "ci95_low": lo,
        "ci95_high": hi,
        "users": len(paired_ndcg),
    })

    # Warm-tail metrics: only the established 931-user warm-tail scope.
    baseline_tail = baseline_all.loc[warm_users]
    current_tail = current.loc[warm_users]

    for col, label in [
        ("warm_tail_recall@10", "Warm-tail Recall@10 change"),
        ("warm_tail_hit_rate@10", "Warm-tail HitRate@10 change"),
    ]:
        differences = (
            current_tail[col].to_numpy()
            - baseline_tail[col].to_numpy()
        )
        mean, lo, hi = bootstrap_mean_ci(differences)

        ci_rows.append({
            "reserved_slots": k,
            "metric": label,
            "mean_change": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "users": len(differences),
        })

ci_table = pd.DataFrame(ci_rows)
display(ci_table.round(6))


,reserved_slots,metric,mean_change,ci95_low,ci95_high,users
0,1,Overall NDCG@10 change,-0.002976,-0.003529,-0.002436,5661
1,1,Warm-tail Recall@10 change,0.008746,0.004117,0.014504,931
2,1,Warm-tail HitRate@10 change,0.013963,0.006445,0.022556,931
3,2,Overall NDCG@10 change,-0.006403,-0.007189,-0.005624,5661
4,2,Warm-tail Recall@10 change,0.017912,0.010777,0.025946,931
5,2,Warm-tail HitRate@10 change,0.029001,0.018260,0.039742,931
6,3,Overall NDCG@10 change,-0.010177,-0.011197,-0.009158,5661
7,3,Warm-tail Recall@10 change,0.025073,0.016669,0.034193,931
8,3,Warm-tail HitRate@10 change,0.039742,0.027927,0.052632,931


In [29]:

# User-level trade-off analysis, averaged across the 10 tie-break seeds.

cols = [
    "overall_ndcg@10",
    "warm_tail_recall@10",
    "warm_tail_hit_rate@10",
]

user_level = (
    rerank_results
    .groupby(["reserved_slots", "user_id"])[cols]
    .mean()
)

tail_users = sorted(relevant_by_scope["warm_tail"].keys())
all_users = sorted(overall_relevant.keys())

baseline_all = user_level.loc[0].reindex(all_users)
baseline_tail = user_level.loc[0].reindex(tail_users)

rows = []

for k in [1, 2, 3]:
    current_all = user_level.loc[k].reindex(all_users)
    current_tail = user_level.loc[k].reindex(tail_users)

    # Per-user overall NDCG change.
    delta_ndcg = (
        current_all["overall_ndcg@10"].to_numpy()
        - baseline_all["overall_ndcg@10"].to_numpy()
    )

    # A positive value means the user gained a relevant warm-tail hit
    # in at least one seed, after averaging their hit rate over seeds.
    delta_tail_hit = (
        current_tail["warm_tail_hit_rate@10"].to_numpy()
        - baseline_tail["warm_tail_hit_rate@10"].to_numpy()
    )

    tail_gained = delta_tail_hit > 1e-12
    overall_lost = delta_ndcg < -1e-12
    overall_improved = delta_ndcg > 1e-12
    overall_unchanged = ~(overall_lost | overall_improved)

    rows.append({
        "reserved_slots": k,
        "overall_users_lost_ndcg": int(overall_lost.sum()),
        "overall_users_unchanged": int(overall_unchanged.sum()),
        "overall_users_gained_ndcg": int(overall_improved.sum()),
        "warm_tail_users_with_a_hit": int(tail_gained.sum()),
        "tail_hit_users_also_lost_ndcg": int(
            (tail_gained & overall_lost[[
                all_users.index(uid) for uid in tail_users
            ]]).sum()
        ),
        "tail_hit_users_not_losing_ndcg": int(
            (
                tail_gained
                & ~overall_lost[[
                    all_users.index(uid) for uid in tail_users
                ]]
            ).sum()
        ),
    })

impact_table = pd.DataFrame(rows)
display(impact_table)


,reserved_slots,overall_users_lost_ndcg,overall_users_unchanged,overall_users_gained_ndcg,warm_tail_users_with_a_hit,tail_hit_users_also_lost_ndcg,tail_hit_users_not_losing_ndcg
0,1,167,5482,12,13,0,13
1,2,340,5294,27,27,0,27
2,3,506,5118,37,37,0,37


In [30]:

# Selective, rank-gated warm-tail reranking.
# Does not use validation relevance to select candidates.
# Candidate quality thresholds are exploratory and require independent
# evaluation before making a final claim.

TAIL_MODEL = "LightFM_content_v1"
gate_values = [0, 10, 25, 50, 100, 250, 500, 1000, 2000, 5000, 17241]

warm_users = set(relevant_by_scope["warm_tail"].keys())
tail_candidate_ids = np.flatnonzero(
    np.asarray(warm_tail_mask, dtype=bool)
    & np.asarray(item_has_als_factors, dtype=bool)
)

# Step A: identify each user's best warm-tail candidate outside their
# existing ALS top 10, and its rank in Content-v1's eligible full catalogue.
candidate_meta = {}

for user_id in eval_users:
    row = np.asarray(
        score_functions[TAIL_MODEL]([user_id])
    )[0].astype(np.float64, copy=True)

    # Match the existing fallback policy for users without positive history.
    if not u_has_pos[user_id]:
        row = np.asarray(
            popularity_fallback_scores(user_id),
            dtype=np.float64
        ).copy()

    row[seen_train[user_id].indices] = -np.inf

    valid_tail = tail_candidate_ids[np.isfinite(row[tail_candidate_ids])]

    if len(valid_tail):
        # We only need the top 11 tail candidates: ALS already contains
        # at most 10 movies, so one of these must be outside the ALS top 10.
        n_top = min(11, len(valid_tail))
        tail_scores = row[valid_tail]
        top_idx = np.argpartition(tail_scores, -n_top)[-n_top:]
        top_order = valid_tail[
            top_idx[np.argsort(tail_scores[top_idx])[::-1]]
        ]
    else:
        top_order = np.array([], dtype=int)

    for seed in range(10):
        base = np.asarray(
            recommendations["ALS_only"][seed][user_id],
            dtype=np.int32
        )
        base_set = set(map(int, base))

        candidate = next(
            (int(item) for item in top_order if int(item) not in base_set),
            None
        )

        if candidate is None:
            candidate_rank = None
        else:
            candidate_score = row[candidate]
            candidate_rank = 1 + int(
                np.sum(np.isfinite(row) & (row > candidate_score))
            )

        candidate_meta[(seed, user_id)] = (candidate, candidate_rank)


# Step B: evaluate each rank gate over the same 10 ALS recommendation seeds.
gate_rows = []

for gate in gate_values:
    for seed in range(10):
        gated_recs = {}
        injected_users = set()

        for user_id in eval_users:
            base = np.asarray(
                recommendations["ALS_only"][seed][user_id],
                dtype=np.int32
            )

            candidate, candidate_rank = candidate_meta[(seed, user_id)]

            if (
                gate > 0
                and candidate is not None
                and candidate_rank is not None
                and candidate_rank <= gate
            ):
                gated_recs[user_id] = np.concatenate(
                    [base[:-1], np.array([candidate], dtype=np.int32)]
                )
                injected_users.add(user_id)
            else:
                gated_recs[user_id] = base.copy()

        overall_m = ranking_metrics_at_10(gated_recs, overall_relevant)
        tail_m = ranking_metrics_at_10(gated_recs, relevant_by_scope["warm_tail"])

        for user_id in eval_users:
            gate_rows.append({
                "max_content_rank": gate,
                "seed": seed,
                "user_id": user_id,
                "overall_ndcg": overall_m.loc[user_id, "ndcg@10"],
                "overall_recall": overall_m.loc[user_id, "recall@10"],
                "overall_hit_rate": overall_m.loc[user_id, "hit_rate@10"],
                "warm_tail_recall": (
                    tail_m.loc[user_id, "recall@10"]
                    if user_id in warm_users else np.nan
                ),
                "warm_tail_hit_rate": (
                    tail_m.loc[user_id, "hit_rate@10"]
                    if user_id in warm_users else np.nan
                ),
                "injected": user_id in injected_users,
            })

gate_results = pd.DataFrame(gate_rows)

# Average each user's outcome across the 10 seeds before summarizing users.
per_user_gate = (
    gate_results
    .groupby(["max_content_rank", "user_id"], as_index=False)
    .agg(
        overall_ndcg=("overall_ndcg", "mean"),
        overall_recall=("overall_recall", "mean"),
        overall_hit_rate=("overall_hit_rate", "mean"),
        warm_tail_recall=("warm_tail_recall", "mean"),
        warm_tail_hit_rate=("warm_tail_hit_rate", "mean"),
        injection_frequency=("injected", "mean"),
    )
)

summary_rows = []

for gate in gate_values:
    group = per_user_gate[per_user_gate["max_content_rank"] == gate]
    tail_group = group[group["user_id"].isin(warm_users)]

    summary_rows.append({
        "max_content_rank": gate,
        "overall_ndcg": group["overall_ndcg"].mean(),
        "overall_recall": group["overall_recall"].mean(),
        "overall_hit_rate": group["overall_hit_rate"].mean(),
        "warm_tail_recall": tail_group["warm_tail_recall"].mean(),
        "warm_tail_hit_rate": tail_group["warm_tail_hit_rate"].mean(),
        "users_injected_any_seed": int(
            (group["injection_frequency"] > 0).sum()
        ),
        "user_seed_injection_rate": group["injection_frequency"].mean(),
    })

gate_summary = pd.DataFrame(summary_rows).set_index("max_content_rank")

baseline_ndcg = gate_summary.loc[0, "overall_ndcg"]
baseline_tail_recall = gate_summary.loc[0, "warm_tail_recall"]
baseline_tail_hit = gate_summary.loc[0, "warm_tail_hit_rate"]

gate_summary["overall_ndcg_loss_pct"] = (
    (baseline_ndcg - gate_summary["overall_ndcg"])
    / baseline_ndcg * 100
)
gate_summary["warm_tail_recall_gain"] = (
    gate_summary["warm_tail_recall"] - baseline_tail_recall
)
gate_summary["warm_tail_hit_rate_gain"] = (
    gate_summary["warm_tail_hit_rate"] - baseline_tail_hit
)

display(gate_summary.round(6))


,overall_ndcg,overall_recall,overall_hit_rate,warm_tail_recall,warm_tail_hit_rate,users_injected_any_seed,user_seed_injection_rate,overall_ndcg_loss_pct,warm_tail_recall_gain,warm_tail_hit_rate_gain
max_content_rank,,,,,,,,,,
0,0.081462,0.106220,0.302067,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.000000
10,0.081448,0.106194,0.301890,0.000000,0.000000,19,0.003356,0.017230,0.000000,0.000000
25,0.081448,0.106194,0.301890,0.000000,0.000000,36,0.006359,0.017230,0.000000,0.000000
50,0.081448,0.106194,0.301890,0.000000,0.000000,74,0.013072,0.017230,0.000000,0.000000
100,0.081407,0.106093,0.301537,0.000153,0.001074,155,0.027380,0.067400,0.000153,0.001074
250,0.081283,0.105807,0.301007,0.001586,0.003222,498,0.087970,0.219694,0.001586,0.003222
500,0.081007,0.105221,0.300300,0.004629,0.008593,1273,0.224872,0.558150,0.004629,0.008593
1000,0.080264,0.103449,0.296767,0.007672,0.012889,3055,0.539657,1.471136,0.007672,0.012889
2000,0.078607,0.099668,0.289525,0.008746,0.013963,5473,0.966790,3.504927,0.008746,0.013963


In [31]:

# Paired bootstrap CIs for selective reranking gates.
# User-level metrics are already averaged across the 10 tie-break seeds.

rng_ci = np.random.default_rng(2027)
N_BOOT = 3000
BATCH_SIZE = 100

gates_to_test = [100, 250, 500, 1000, 2000]

all_users = sorted(overall_relevant.keys())
warm_users = sorted(relevant_by_scope["warm_tail"].keys())

user_tables = {
    gate: (
        per_user_gate[
            per_user_gate["max_content_rank"] == gate
        ]
        .set_index("user_id")
        .reindex(all_users)
    )
    for gate in [0] + gates_to_test
}

def paired_bootstrap_ci(differences):
    differences = np.asarray(differences, dtype=float)
    n = len(differences)
    boot_means = np.empty(N_BOOT)

    for start in range(0, N_BOOT, BATCH_SIZE):
        batch_n = min(BATCH_SIZE, N_BOOT - start)
        indices = rng_ci.integers(0, n, size=(batch_n, n))
        boot_means[start:start + batch_n] = (
            differences[indices].mean(axis=1)
        )

    return (
        float(differences.mean()),
        float(np.quantile(boot_means, 0.025)),
        float(np.quantile(boot_means, 0.975)),
    )

baseline = user_tables[0]
ci_rows = []

comparisons = [
    (
        "overall_ndcg",
        "Overall NDCG@10 change",
        all_users,
    ),
    (
        "warm_tail_recall",
        "Warm-tail Recall@10 gain",
        warm_users,
    ),
    (
        "warm_tail_hit_rate",
        "Warm-tail HitRate@10 gain",
        warm_users,
    ),
]

for gate in gates_to_test:
    current = user_tables[gate]

    for col, label, users in comparisons:
        delta = (
            current.loc[users, col].to_numpy(dtype=float)
            - baseline.loc[users, col].to_numpy(dtype=float)
        )

        mean, low, high = paired_bootstrap_ci(delta)

        ci_rows.append({
            "gate": gate,
            "metric": label,
            "mean_change": mean,
            "ci95_low": low,
            "ci95_high": high,
            "users": len(users),
        })

selective_ci = pd.DataFrame(ci_rows)
display(selective_ci.round(6))


,gate,metric,mean_change,ci95_low,ci95_high,users
0,100,Overall NDCG@10 change,-0.000055,-0.000142,0.000013,5661
1,100,Warm-tail Recall@10 gain,0.000153,0.000000,0.000460,931
2,100,Warm-tail HitRate@10 gain,0.001074,0.000000,0.003222,931
3,250,Overall NDCG@10 change,-0.000179,-0.000328,-0.000052,5661
4,250,Warm-tail Recall@10 gain,0.001586,0.000000,0.003941,931
5,250,Warm-tail HitRate@10 gain,0.003222,0.000000,0.007519,931
6,500,Overall NDCG@10 change,-0.000455,-0.000732,-0.000199,5661
7,500,Warm-tail Recall@10 gain,0.004629,0.001432,0.008593,931
8,500,Warm-tail HitRate@10 gain,0.008593,0.003222,0.015038,931
9,1000,Overall NDCG@10 change,-0.001198,-0.001604,-0.000827,5661


In [32]:

# Find potentially relevant split/data variables without printing their contents.
keywords = (
    "test", "holdout", "split", "valid", "train",
    "rating", "cutoff", "timestamp", "date"
)

for name in sorted(globals()):
    if not any(word in name.lower() for word in keywords):
        continue

    obj = globals()[name]
    shape = getattr(obj, "shape", None)
    columns = getattr(obj, "columns", None)

    # Keep the inventory manageable.
    if shape is not None or columns is not None:
        print(
            f"{name}: type={type(obj).__name__}, "
            f"shape={shape}, "
            f"columns={list(columns)[:12] if columns is not None else None}"
        )


candidate_frame: type=DataFrame, shape=(931, 4), columns=['ndcg@10', 'recall@10', 'hit_rate@10', 'hit_count@10']
candidate_order: type=ndarray, shape=(17241,), columns=None
candidate_overall_values: type=ndarray, shape=(5661,), columns=None
candidate_score: type=float64, shape=(), columns=None
candidate_values: type=ndarray, shape=(931,), columns=None
interactions_test: type=coo_matrix, shape=(2, 3), columns=None
seen_train: type=csr_matrix, shape=(20093, 17241), columns=None
tail_candidate_ids: type=ndarray, shape=(10502,), columns=None
train: type=DataFrame, shape=(2919680, 4), columns=['u', 'i', 'rating', 'timestamp']
train_positive: type=DataFrame, shape=(1445945, 2), columns=['u', 'i']
valid_tail: type=ndarray, shape=(10482,), columns=None
valid_targets: type=ndarray, shape=(1,), columns=None


In [33]:
print("=== Interaction DataFrames available in memory ===")

found = 0

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    cols = set(map(str, obj.columns))

    # Recognize common MovieLens interaction schemas.
    is_interactions = (
        {"u", "i", "rating", "timestamp"}.issubset(cols)
        or {"userId", "movieId", "rating", "timestamp"}.issubset(cols)
        or {"user_id", "movie_id", "rating", "timestamp"}.issubset(cols)
    )

    if not is_interactions:
        continue

    found += 1
    timestamp_col = next(
        c for c in obj.columns
        if str(c).lower() == "timestamp"
    )

    print(f"\n{name}")
    print("  Shape:", obj.shape)
    print("  Columns:", obj.columns.tolist())
    print("  Timestamp range:",
          obj[timestamp_col].min(),
          "to",
          obj[timestamp_col].max())

    user_col = next(
        c for c in obj.columns
        if str(c).lower() in {"u", "userid", "user_id"}
    )
    item_col = next(
        c for c in obj.columns
        if str(c).lower() in {"i", "movieid", "movie_id"}
    )

    print("  Unique users:", obj[user_col].nunique())
    print("  Unique movies:", obj[item_col].nunique())

if found == 0:
    print("No interaction DataFrames found.")


=== Interaction DataFrames available in memory ===

train
  Shape: (2919680, 4)
  Columns: ['u', 'i', 'rating', 'timestamp']
  Timestamp range: 822873600 to 1697070126
  Unique users: 20093
  Unique movies: 17236

val
  Shape: (137197, 4)
  Columns: ['u', 'i', 'rating', 'timestamp']
  Timestamp range: 825638400 to 1697130034
  Unique users: 20093
  Unique movies: 10192


In [34]:

# Check whether train -> val is chronological for each user.

train_bounds = (
    train.groupby("u")["timestamp"]
    .agg(train_min="min", train_max="max")
)

val_bounds = (
    val.groupby("u")["timestamp"]
    .agg(val_min="min", val_max="max")
)

bounds = train_bounds.join(val_bounds, how="inner")

bounds["chronological"] = bounds["train_max"] <= bounds["val_min"]
bounds["strictly_chronological"] = bounds["train_max"] < bounds["val_min"]

print("Users present in both splits:", len(bounds))
print(
    "Users with all training timestamps <= validation timestamps:",
    int(bounds["chronological"].sum()),
    f"({bounds['chronological'].mean():.2%})"
)
print(
    "Users with strict timestamp separation:",
    int(bounds["strictly_chronological"].sum()),
    f"({bounds['strictly_chronological'].mean():.2%})"
)

print("\nExamples of users whose timestamp ranges overlap:")
display(
    bounds.loc[~bounds["chronological"]]
    .head(10)
)

# Check exact user-item pair overlap across the two splits.
train_pairs = pd.MultiIndex.from_frame(train[["u", "i"]].drop_duplicates())
val_pairs = pd.MultiIndex.from_frame(val[["u", "i"]].drop_duplicates())

print("\nUnique user-item pairs in train:", len(train_pairs))
print("Unique user-item pairs in validation:", len(val_pairs))
print("Pairs appearing in both splits:", len(train_pairs.intersection(val_pairs)))


Users present in both splits: 20093
Users with all training timestamps <= validation timestamps: 20093 (100.00%)
Users with strict timestamp separation: 16672 (82.97%)

Examples of users whose timestamp ranges overlap:


,train_min,train_max,val_min,val_max,chronological,strictly_chronological
u,,,,,,



Unique user-item pairs in train: 2919680
Unique user-item pairs in validation: 137197
Pairs appearing in both splits: 0


In [35]:
from pathlib import Path

cwd = Path.cwd()
print("Notebook working directory:", cwd)

# Look for likely interaction datasets in the project and nearby data folders.
roots = {cwd, cwd.parent, cwd.parent.parent}

for root in sorted(roots, key=lambda p: len(str(p))):
    if not root.exists():
        continue

    print(f"\n--- Data files under {root} ---")

    for folder_name in ["data", "datasets", "dataset"]:
        folder = root / folder_name
        if not folder.is_dir():
            continue

        for pattern in [
            "*.csv", "*.csv.gz", "*.parquet", "*.feather",
            "*.pkl", "*.pickle", "*.zip"
        ]:
            for path in folder.rglob(pattern):
                try:
                    size_mb = path.stat().st_size / (1024 ** 2)
                    print(f"{size_mb:8.1f} MB  {path}")
                except OSError:
                    pass

Notebook working directory: /mnt/c/Users/Lenovo/Documents/movie-recommender/notebooks

--- Data files under /mnt/c/Users/Lenovo/Documents ---

--- Data files under /mnt/c/Users/Lenovo/Documents/movie-recommender ---
    19.1 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/interim/wikidata_values_full.csv
     0.0 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/interim/sample_10pct/tmdb_high_priority_mapping_review.csv
     0.0 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/interim/sample_10pct/tmdb_imdb_id_mismatch_review.csv
     0.0 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/interim/sample_10pct/tmdb_year_mismatch_review.csv
     1.9 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/raw/ml-32m/links.csv
     4.0 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/raw/ml-32m/movies.csv
   836.4 MB  /mnt/c/Users/Lenovo/Documents/movie-recommender/data/raw/ml-32m/ratings.csv
    69.0 MB  /mnt/c/Users/Lenovo/Documents/movie-recommende

## Audit the existing file

In [36]:
from pathlib import Path
import pandas as pd

test_path = Path("../data/processed/sample_10pct/test.parquet")
test = pd.read_parquet(test_path)

print("=== Split sizes ===")
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(
        f"{name}: rows={len(df):,}, "
        f"users={df['u'].nunique():,}, "
        f"movies={df['i'].nunique():,}, "
        f"timestamps={df['timestamp'].min()} to {df['timestamp'].max()}"
    )

# Check the per-user chronology across all three splits.
train_bounds = train.groupby("u")["timestamp"].agg(train_max="max")
val_bounds = val.groupby("u")["timestamp"].agg(
    val_min="min", val_max="max"
)
test_bounds = test.groupby("u")["timestamp"].agg(test_min="min")

bounds = train_bounds.join(val_bounds, how="inner").join(
    test_bounds, how="inner"
)

train_before_val = bounds["train_max"] <= bounds["val_min"]
val_before_test = bounds["val_max"] <= bounds["test_min"]

print("\n=== Per-user chronology ===")
print("Users present in all three splits:", len(bounds))
print(
    "Train ends before/equal to validation begins:",
    f"{train_before_val.sum():,}/{len(bounds):,}"
)
print(
    "Validation ends before/equal to test begins:",
    f"{val_before_test.sum():,}/{len(bounds):,}"
)
print(
    "Strict validation-to-test separation:",
    f"{(bounds['val_max'] < bounds['test_min']).sum():,}/{len(bounds):,}"
)

# Check user-movie overlap without evaluating recommendation performance.
def pair_index(df):
    pairs = df[["u", "i"]].drop_duplicates()
    return pd.MultiIndex.from_frame(pairs)

train_pairs = pair_index(train)
val_pairs = pair_index(val)
test_pairs = pair_index(test)

print("\n=== User-movie pair overlap ===")
print("Train ∩ validation:", len(train_pairs.intersection(val_pairs)))
print("Train ∩ test:", len(train_pairs.intersection(test_pairs)))
print("Validation ∩ test:", len(val_pairs.intersection(test_pairs)))

=== Split sizes ===
train: rows=2,919,680, users=20,093, movies=17,236, timestamps=822873600 to 1697070126
val: rows=137,197, users=20,093, movies=10,192, timestamps=825638400 to 1697130034
test: rows=137,197, users=20,093, movies=10,776, timestamps=827224187 to 1697161415

=== Per-user chronology ===
Users present in all three splits: 20093
Train ends before/equal to validation begins: 20,093/20,093
Validation ends before/equal to test begins: 20,093/20,093
Strict validation-to-test separation: 16,890/20,093

=== User-movie pair overlap ===
Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0


In [37]:

import inspect

print("=== Scoring function implementations ===")

for name, fn in score_functions.items():
    print(f"\n--- {name} ---")
    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError):
        print("Source unavailable:", fn.__name__)

    closure = inspect.getclosurevars(fn)
    captured = {**closure.globals, **closure.nonlocals}

    print("Relevant captured objects:")
    shown = set()

    for key, obj in captured.items():
        low = key.lower()
        cls = type(obj).__name__.lower()

        relevant = (
            "als" in low
            or "lightfm" in low
            or "model" in low
            or "factor" in low
            or "feature" in low
            or "interaction" in low
            or "item" in low and hasattr(obj, "shape")
            or "user" in low and hasattr(obj, "shape")
            or "lightfm" in cls
        )

        if relevant and key not in shown:
            shown.add(key)
            print(
                f"  {key}: type={type(obj).__name__}, "
                f"shape={getattr(obj, 'shape', None)}"
            )

print("\n=== Core data and features ===")

for name in [
    "train_positive",
    "seen_train",
    "item_map",
    "X_v1",
    "n_items",
    "SEED",
    "K",
]:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: type={type(obj).__name__}, "
            f"shape={getattr(obj, 'shape', None)}, "
            f"value={obj if np.isscalar(obj) else ''}"
        )
    else:
        print(f"{name}: not found")


=== Scoring function implementations ===

--- ALS_only ---
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int64)
        return (P[ub] @ Q.T).astype(np.float32, copy=False)

Relevant captured objects:

--- LightFM_identity ---
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int32)
        scores = user_vectors[ub] @ item_vectors.T
        scores += user_bias[ub, None]
        scores += item_bias[None, :]
        return scores.astype(np.float32, copy=False)

Relevant captured objects:
  item_bias: type=ndarray, shape=(17241,)
  item_vectors: type=ndarray, shape=(17241, 32)
  user_bias: type=ndarray, shape=(20093,)
  user_vectors: type=ndarray, shape=(20093, 32)

--- LightFM_content_v1 ---
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int32)
        scores = user_vectors[ub] @ item_vectors.T
        scores += user_bias[ub, None]
        scores += item_bias[None, :]
        return scores.astype(np

In [38]:
# Compact model/configuration audit (avoid printing recommendation dictionaries or arrays).
import numpy as np

print("=== Compact model configuration audit ===")
print("ALS fit parameters: k=64, alpha=5.0, lam=80.0, n_iter=10, seed=42")
print("ALS factor shapes:", P_als.shape, Q_als.shape)
print("Feature matrices:")
for name in ["interactions", "item_features_identity", "item_features_v1", "item_features_tmdb_v2"]:
    obj = globals().get(name)
    if obj is not None:
        print(f"  {name}: shape={obj.shape}, nnz={getattr(obj, 'nnz', 'n/a')}")
print("LightFM configuration:")
for name, model in lightfm_models.items():
    print(
        f"  {name}: components={model.no_components}, loss={model.loss}, "
        f"epochs={LIGHTFM_EPOCHS}, learning_rate={model.learning_rate}, "
        f"item_alpha={model.item_alpha}, user_alpha={model.user_alpha}, "
        f"max_sampled={model.max_sampled}"
    )


=== Compact model configuration audit ===
ALS fit parameters: k=64, alpha=5.0, lam=80.0, n_iter=10, seed=42
ALS factor shapes: (20093, 64) (17241, 64)
Feature matrices:
  interactions: shape=(20093, 17241), nnz=1445945
  item_features_identity: shape=(17241, 17241), nnz=17241
  item_features_v1: shape=(17241, 17272), nnz=71200
  item_features_tmdb_v2: shape=(17241, 108569), nnz=890685
LightFM configuration:
  LightFM_identity: components=32, loss=warp, epochs=15, learning_rate=0.05, item_alpha=1e-06, user_alpha=1e-06, max_sampled=10
  LightFM_content_v1: components=32, loss=warp, epochs=15, learning_rate=0.05, item_alpha=1e-06, user_alpha=1e-06, max_sampled=10
  LightFM_TMDB_v2: components=32, loss=warp, epochs=15, learning_rate=0.05, item_alpha=1e-06, user_alpha=1e-06, max_sampled=10


In [39]:

import inspect
import numpy as np

print("=== ALS fitting function ===")
print("Signature:", inspect.signature(fit_implicit_als_train_only))
print(inspect.getsource(fit_implicit_als_train_only))

print("\n=== LightFM fitting function ===")
print("Signature:", inspect.signature(fit_lightfm_candidate))
print(inspect.getsource(fit_lightfm_candidate))

print("\n=== Existing ALS model metadata ===")
print("Keys:", list(als_model.keys()))

for key, value in als_model.items():
    if isinstance(value, dict):
        print(f"{key}: dictionary keys={list(value.keys())}")
    elif isinstance(value, np.ndarray):
        print(f"{key}: array shape={value.shape}, dtype={value.dtype}")
    elif np.isscalar(value) or isinstance(value, (str, bool)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__}")

print("\n=== LightFM model metadata ===")
print("Model keys:", list(lightfm_models.keys()))

for key, model in lightfm_models.items():
    print(f"\n{key}: {type(model).__name__}")
    for attr in [
        "no_components", "loss", "learning_rate",
        "item_alpha", "user_alpha", "max_sampled"
    ]:
        if hasattr(model, attr):
            print(f"  {attr}={getattr(model, attr)}")


=== ALS fitting function ===
Signature: (df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=42)
def fit_implicit_als_train_only(df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED):
    d = df.loc[df["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(dtype=np.int64), d["i"].to_numpy(dtype=np.int64)),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)
    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)
        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(f"ALS iteration {iteration:02d}/{n_iter} | elapsed {time.time() - star

In [40]:

# Build the historical data available immediately before the test period.
train_val = pd.concat([train, val], ignore_index=True)

# Count distinct positive user-item interactions per item.
positive_tv = (
    train_val.loc[train_val["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
)

n_pos_tv = np.bincount(
    positive_tv["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
)

print("=== Existing popularity masks ===")

for name in ["head_mask", "tail_mask", "warm_tail_mask", "strict_cold_mask"]:
    if name not in globals():
        continue

    mask = np.asarray(globals()[name], dtype=bool)
    counts = n_pos[mask]

    print(f"\n{name}")
    print("  Items:", int(mask.sum()))
    print("  Zero-positive items:", int((counts == 0).sum()))

    if len(counts):
        print("  Min/max original positive counts:",
              int(counts.min()), int(counts.max()))

if "tail_mask" in globals():
    old_tail = np.asarray(tail_mask, dtype=bool)
    print("\nOriginal tail boundary:")
    print("  Max count among tail items:",
          int(n_pos[old_tail].max()))
    print("  Min count among non-tail items:",
          int(n_pos[~old_tail].min()))

print("\n=== Before-test fitting data ===")
print("Train + validation rows:", len(train_val))
print("Unique positive user-item pairs:", len(positive_tv))
print("Catalogue items with positive history:", int((n_pos_tv > 0).sum()))
print("Catalogue items without positive history:", int((n_pos_tv == 0).sum()))

print("\n=== New positive-count distribution ===")
print(pd.Series(n_pos_tv).describe(
    percentiles=[0.25, 0.50, 0.75, 0.80, 0.90, 0.95, 0.99]
).round(2))


=== Existing popularity masks ===

tail_mask
  Items: 11597
  Zero-positive items: 1095
  Min/max original positive counts: 0 19

warm_tail_mask
  Items: 10502
  Zero-positive items: 0
  Min/max original positive counts: 1 19

strict_cold_mask
  Items: 1095
  Zero-positive items: 1095
  Min/max original positive counts: 0 0

Original tail boundary:
  Max count among tail items: 19
  Min count among non-tail items: 20

=== Before-test fitting data ===
Train + validation rows: 3056877
Unique positive user-item pairs: 1521298
Catalogue items with positive history: 16220
Catalogue items without positive history: 1021

=== New positive-count distribution ===
count    17241.00
mean        88.24
std        340.27
min          0.00
25%          3.00
50%          8.00
75%         36.00
80%         56.00
90%        172.00
95%        416.00
99%       1497.20
max       8639.00
dtype: float64


In [41]:

import scipy.sparse as sp

# Fit only on data available before the test period.
train_val = pd.concat([train, val], ignore_index=True)

# Positive interactions, deduplicated at user-item level.
positive_tv = (
    train_val.loc[train_val["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
    .reset_index(drop=True)
)

# LightFM input: binary positive interactions.
interactions_tv = sp.coo_matrix(
    (
        np.ones(len(positive_tv), dtype=np.float32),
        (
            positive_tv["u"].to_numpy(dtype=np.int32),
            positive_tv["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
)

# All historical interactions, including ratings below the relevance threshold.
# These items must be excluded from future recommendations.
seen_pairs_tv = (
    train_val[["u", "i"]]
    .drop_duplicates(["u", "i"])
)

seen_tv = sp.csr_matrix(
    (
        np.ones(len(seen_pairs_tv), dtype=np.float32),
        (
            seen_pairs_tv["u"].to_numpy(dtype=np.int32),
            seen_pairs_tv["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
)
seen_tv.sum_duplicates()
seen_tv.data[:] = 1.0

# Popularity is calculated strictly from positive interactions in train + val.
n_pos_tv = np.bincount(
    positive_tv["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
)

# Freeze the original absolute boundary: at most 19 positive interactions.
tail_mask_tv = n_pos_tv <= 19
warm_tail_mask_tv = (n_pos_tv >= 1) & (n_pos_tv <= 19)
strict_cold_mask_tv = n_pos_tv == 0
item_has_als_factors_tv = n_pos_tv > 0

u_has_pos_tv = np.zeros(n_users, dtype=bool)
u_has_pos_tv[positive_tv["u"].unique()] = True

print("=== Prepared pre-test training inputs ===")
print("Train + validation rows:", f"{len(train_val):,}")
print("Unique positive user-item pairs:", f"{len(positive_tv):,}")
print("LightFM interaction matrix:", interactions_tv.shape,
      "| nnz:", f"{interactions_tv.nnz:,}")
print("Seen-history matrix:", seen_tv.shape,
      "| nnz:", f"{seen_tv.nnz:,}")

print("\n=== Frozen tail definition (positive history <= 19) ===")
print("Tail items, including strict-cold:", int(tail_mask_tv.sum()))
print("Warm-tail items:", int(warm_tail_mask_tv.sum()))
print("Strict-cold items:", int(strict_cold_mask_tv.sum()))
print("Items with positive history:", int(item_has_als_factors_tv.sum()))
print("Users with positive history:", int(u_has_pos_tv.sum()))


=== Prepared pre-test training inputs ===
Train + validation rows: 3,056,877
Unique positive user-item pairs: 1,521,298
LightFM interaction matrix: (20093, 17241) | nnz: 1,521,298
Seen-history matrix: (20093, 17241) | nnz: 3,056,877

=== Frozen tail definition (positive history <= 19) ===
Tail items, including strict-cold: 11456
Warm-tail items: 10435
Strict-cold items: 1021
Items with positive history: 16220
Users with positive history: 20065


In [42]:

# Inspect previously executed notebook code to recover the evaluation protocol.
# This does not calculate any test metrics.

keywords = [
    "relevant_by_scope",
    "valid_targets",
    "val_targets",
    "warm_tail",
    "strict_cold",
    "2016",
    "timestamp",
    "rating",
]

history = globals().get("In", [])
matches = []

for cell_num, cell_source in enumerate(history):
    if not isinstance(cell_source, str):
        continue

    lines = cell_source.splitlines()

    matching_lines = [
        i for i, line in enumerate(lines)
        if any(keyword.lower() in line.lower() for keyword in keywords)
    ]

    if matching_lines:
        # Display only useful cells, with nearby context.
        selected = set()
        for i in matching_lines:
            selected.update(range(max(0, i - 2), min(len(lines), i + 3)))

        excerpt = "\n".join(
            f"{i + 1:>3}: {lines[i]}"
            for i in sorted(selected)
        )

        matches.append((cell_num, excerpt))

print("Matched notebook cells:", len(matches))

for cell_num, excerpt in matches[-12:]:
    print(f"\n{'=' * 60}")
    print(f"INPUT CELL {cell_num}")
    print(excerpt)


Matched notebook cells: 27

INPUT CELL 27
 10: 
 11: # Warm-tail metrics: only users in the established warm-tail evaluation scope.
 12: warm_tail_users = set(relevant_by_scope["warm_tail"].keys())
 13: warm_results = rerank_results[
 14:     rerank_results["user_id"].isin(warm_tail_users)
 15: ]
 16: 
 18:     warm_results.groupby("reserved_slots")
 19:     .agg(
 20:         warm_tail_users=("user_id", "nunique"),
 21:         warm_tail_ndcg=("warm_tail_ndcg@10", "mean"),
 22:         warm_tail_recall=("warm_tail_recall@10", "mean"),
 23:         warm_tail_hit_rate=("warm_tail_hit_rate@10", "mean"),
 24:     )
 25: )
 35: )
 36: 
 37: baseline_tail_recall = corrected_summary.loc[0, "warm_tail_recall"]
 38: baseline_tail_hit = corrected_summary.loc[0, "warm_tail_hit_rate"]
 39: 
 40: corrected_summary["tail_recall_gain"] = (
 41:     corrected_summary["warm_tail_recall"] - baseline_tail_recall
 42: )
 43: corrected_summary["tail_hit_rate_gain"] = (
 44:     corrected_summary["warm_tai

In [43]:

import re

history = globals().get("In", [])

patterns = [
    r"^\s*(?:overall_relevant|relevant_by_scope|valid_targets|val_targets|test_targets)\s*=",
    r"^\s*(?:REL|RELEVANCE_THRESHOLD|RATING_THRESHOLD)\s*=",
    r"2016",
    r"relevant_by_scope\s*\[",
]

found_cells = []

for cell_num, source in enumerate(history):
    if not isinstance(source, str):
        continue

    matching_lines = [
        i for i, line in enumerate(source.splitlines())
        if any(re.search(pattern, line, re.IGNORECASE) for pattern in patterns)
    ]

    if matching_lines:
        found_cells.append((cell_num, source, matching_lines))

print("Candidate source cells:", len(found_cells))

# Prioritize cells that DEFINE the relevance variables.
definitions = [
    entry for entry in found_cells
    if re.search(
        r"^\s*(overall_relevant|relevant_by_scope|valid_targets|val_targets|test_targets)\s*=",
        entry[1],
        re.MULTILINE,
    )
]

for cell_num, source, matching_lines in definitions[:10]:
    lines = source.splitlines()
    selected = set()

    for i in matching_lines:
        selected.update(range(max(0, i - 3), min(len(lines), i + 6)))

    print(f"\n{'=' * 60}\nINPUT CELL {cell_num}")
    print("\n".join(
        f"{i + 1:>3}: {lines[i]}"
        for i in sorted(selected)
    ))

if not definitions:
    print("\nNo direct definitions found. Showing the earliest matching cells instead.")
    for cell_num, source, matching_lines in found_cells[:8]:
        lines = source.splitlines()
        print(f"\n{'=' * 60}\nINPUT CELL {cell_num}")
        for i in range(min(len(lines), 80)):
            print(f"{i + 1:>3}: {lines[i]}")


Candidate source cells: 14

INPUT CELL 2
 75:     }
 76: 
 77: item_idx_val = val_window["i"].to_numpy(dtype=np.int64)
 78: relevant_by_scope = {
 79:     "overall": relevant_dict(val_window),
 80:     "tail": relevant_dict(val_window.loc[tail_mask[item_idx_val]]),
 81:     "strict_cold": relevant_dict(val_window.loc[strict_cold_mask[item_idx_val]]),
 82:     "warm_tail": relevant_dict(val_window.loc[warm_tail_mask[item_idx_val]]),
 83: }
 96: 
 97: print(f"Users: {n_users:,} | Catalogue movies: {n_items:,}")
 98: print(f"Train interactions: {len(train):,} | Positive train pairs (rating >= {REL}): {len(train_positive):,}")
 99: print(f"Validation rows: {len(val):,} | Evaluated post-2016 positive pairs: {len(val_window):,}")
100: print(f"content-v1: {X_v1.shape}, nnz={X_v1.nnz:,}")
101: print(f"TMDB-v2:    {X_tmdb_v2.shape}, nnz={X_tmdb_v2.nnz:,}")
102: print(f"Tail (<20 positive train pairs): {int(tail_mask.sum()):,} movies")
103: print(f"Strict cold (0 positive train pairs): {int(stri

In [44]:
import inspect

print("=== Relevance threshold ===")
print("REL =", REL)

print("\n=== relevant_dict definition ===")
print(inspect.getsource(relevant_dict))

print("\n=== Original evaluation setup (cell 2, lines 1–85) ===")
src = In[2]
for line_no, line in enumerate(src.splitlines()[:85], start=1):
    print(f"{line_no:>3}: {line}")

=== Relevance threshold ===
REL = 4.0

=== relevant_dict definition ===
def relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }


=== Original evaluation setup (cell 2, lines 1–85) ===
  1: # Load train/validation data only; do not read S["test"].
  2: S = load_split(NAME)
  3: train = S["train"].copy()
  4: val = S["val"].copy()
  5: item_map = S["item_map"].sort_values("i").reset_index(drop=True)
  6: user_map = S["user_map"].sort_values("u").reset_index(drop=True)
  7: n_users = int(user_map["u"].max()) + 1
  8: n_items = int(item_map["i"].max()) + 1
  9: 
 10: feature_dir = _PROJECT_ROOT / "data" / "interim" / "hybrid_features"
 11: tmdb_dir = _PROJECT_ROOT / "data" / "processed" / NAME
 12: v1_path = feature_dir / "content_v1.npz"
 13: tmdb_path = tmdb_dir / "tmdb_content_v2.npz"
 14: tmdb_index_path = tmdb_dir / "tmdb_content_v2_index.parquet"
 15: 
 16: for _path in (v1_path, t

In [45]:
import pandas as pd

print("CUTOFF_TS:", CUTOFF_TS)
print(
    "Cutoff date (UTC):",
    pd.to_datetime(CUTOFF_TS, unit="s", utc=True)
)
print("Relevance threshold:", REL)

CUTOFF_TS: 1451606400
Cutoff date (UTC): 2016-01-01 00:00:00+00:00
Relevance threshold: 4.0


In [46]:

import time
import numpy as np
from lightfm import LightFM

# Confirm the new training inputs and feature matrix.
assert len(train_val) > len(train)
assert interactions_tv.shape == (n_users, n_items)
assert item_features_v1.shape[0] == n_items
assert warm_tail_mask_tv.shape == (n_items,)
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))

# --------------------------------------------------
# 1. Refit implicit ALS on train + validation
# --------------------------------------------------
print("=== Fitting ALS on train + validation ===")

start = time.time()

als_final = fit_implicit_als_train_only(
    train_val,
    k=64,
    alpha=5.0,
    lam=80.0,
    n_iter=10,
    seed=42,
)

P_final = als_final["P"]
Q_final = als_final["Q"]

assert P_final.shape == (n_users, 64)
assert Q_final.shape == (n_items, 64)
assert np.isfinite(P_final).all() and np.isfinite(Q_final).all()

print(f"ALS complete in {(time.time() - start) / 60:.2f} minutes")
print("User factors:", P_final.shape)
print("Item factors:", Q_final.shape)

# --------------------------------------------------
# 2. Refit LightFM Content-v1 on train + validation
# --------------------------------------------------
print("\n=== Fitting LightFM Content-v1 on train + validation ===")

lightfm_final = LightFM(
    no_components=LIGHTFM_COMPONENTS,       # 32
    loss="warp",
    learning_rate=LIGHTFM_LEARNING_RATE,    # 0.05
    item_alpha=LIGHTFM_ITEM_ALPHA,           # 1e-6
    user_alpha=LIGHTFM_USER_ALPHA,           # 1e-6
    max_sampled=LIGHTFM_MAX_SAMPLED,         # 10
    random_state=SEED,                       # 42
)

start = time.time()

lightfm_final.fit(
    interactions_tv,
    item_features=item_features_v1,
    epochs=LIGHTFM_EPOCHS,                   # 15
    num_threads=NUM_THREADS,                 # 1
    verbose=True,
)

print(
    f"LightFM Content-v1 complete in "
    f"{(time.time() - start) / 60:.2f} minutes"
)

assert lightfm_final.user_embeddings.shape[0] == n_users
assert np.isfinite(lightfm_final.user_embeddings).all()
assert np.isfinite(lightfm_final.item_embeddings).all()

print("User embeddings:", lightfm_final.user_embeddings.shape)
print("Item-feature embeddings:", lightfm_final.item_embeddings.shape)

# --------------------------------------------------
# 3. Prepare the fallback popularity distribution
# --------------------------------------------------
# Preserve the existing rule: users without positive history receive
# a popularity-based fallback, based on all pre-test interactions.
popularity_scores_tv = np.bincount(
    train_val["i"].to_numpy(dtype=np.int64),
    minlength=n_items,
).astype(np.float64)

print("\n=== Final models ready ===")
print("ALS factors and LightFM model fitted.")
print("Fallback popularity scores:", popularity_scores_tv.shape)
print("Positive-history users:", int(u_has_pos_tv.sum()))
print("Warm-tail catalogue items:", int(warm_tail_mask_tv.sum()))
print("Strict-cold catalogue items:", int(strict_cold_mask_tv.sum()))


=== Fitting ALS on train + validation ===
ALS iteration 01/10 | elapsed 3.9s
ALS iteration 02/10 | elapsed 7.9s
ALS iteration 04/10 | elapsed 16.9s
ALS iteration 06/10 | elapsed 27.1s
ALS iteration 08/10 | elapsed 36.9s
ALS iteration 10/10 | elapsed 45.8s
ALS complete in 0.77 minutes
User factors: (20093, 64)
Item factors: (17241, 64)

=== Fitting LightFM Content-v1 on train + validation ===
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM Content-v1 complete in 0.73 minutes
User embeddings: (20093, 32)
Item-feature embeddings: (17272, 32)

=== Final models ready ===
ALS factors and LightFM model fitted.
Fallback popularity scores: (17241,)
Positive-history users: 20065
Warm-tail catalogue items: 10435
Strict-cold catalogue items: 1021


In [47]:

import numpy as np
import pandas as pd

# ============================================================
# 0. Freeze evaluation settings
# ============================================================
K_EVAL = 10
TIE_SEEDS = range(10)
GATES = [0, 500, 1000]  # 0 = ALS baseline
BATCH_SIZE = 64
BOOTSTRAP_REPS = 3000

# ============================================================
# 1. Construct test relevance using the locked protocol
#    rating >= 4.0 and timestamp >= 2016-01-01 UTC
# ============================================================
test_ts = pd.to_numeric(test["timestamp"], errors="raise")

test_window = test.loc[
    (test["rating"] >= REL) & (test_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

test_item_ids = test_window["i"].to_numpy(dtype=np.int64)

test_scope_masks = {
    "overall": np.ones(len(test_window), dtype=bool),
    "tail": tail_mask_tv[test_item_ids],
    "strict_cold": strict_cold_mask_tv[test_item_ids],
    "warm_tail": warm_tail_mask_tv[test_item_ids],
}

def build_relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

test_relevant = {
    scope: build_relevant_dict(test_window.loc[mask])
    for scope, mask in test_scope_masks.items()
}

test_users = sorted(test_relevant["overall"])
test_warm_users = sorted(test_relevant["warm_tail"])

assert test_users, "No positive test targets found under the locked protocol."
assert len(test_users) == len(test_relevant["overall"])
assert all(
    not (set(items) & set(seen_tv[user].indices))
    for user, items in test_relevant["overall"].items()
), "Found a test target in the user's pre-test history."

print("=== Locked test targets ===")
print("Post-cutoff positive user-movie pairs:", len(test_window))
for scope in ["overall", "tail", "warm_tail", "strict_cold"]:
    pairs = sum(len(v) for v in test_relevant[scope].values())
    print(
        f"{scope}: {pairs:,} relevant pairs across "
        f"{len(test_relevant[scope]):,} users"
    )

# ============================================================
# 2. Create scoring representations from the final models
# ============================================================
als_user_factors = P_final
als_item_factors = Q_final

lf_user_bias, lf_user_vectors = lightfm_final.get_user_representations()
lf_item_bias, lf_item_vectors = lightfm_final.get_item_representations(
    item_features_v1
)

assert als_user_factors.shape == (n_users, 64)
assert als_item_factors.shape == (n_items, 64)
assert lf_user_vectors.shape[0] == n_users
assert lf_item_vectors.shape[0] == n_items

def score_als_batch(user_ids):
    user_ids = np.asarray(user_ids, dtype=np.int64)
    return (als_user_factors[user_ids] @ als_item_factors.T).astype(
        np.float32, copy=False
    )

def score_content_batch(user_ids):
    user_ids = np.asarray(user_ids, dtype=np.int32)
    scores = lf_user_vectors[user_ids] @ lf_item_vectors.T
    scores += lf_user_bias[user_ids, None]
    scores += lf_item_bias[None, :]
    return scores.astype(np.float32, copy=False)

# Same recommendation eligibility and seeded tie-break rules used previously.
def final_top_k(scores, user_id, tie_seed):
    row = np.asarray(scores, dtype=np.float64).copy()

    if not u_has_pos_tv[user_id]:
        row = popularity_scores_tv.copy()
    else:
        row[~item_has_als_factors_tv] = -np.inf

    row[seen_tv[user_id].indices] = -np.inf
    eligible = np.isfinite(row)

    if int(eligible.sum()) < K_EVAL:
        raise RuntimeError(
            f"Only {int(eligible.sum())} eligible items for user {user_id}"
        )

    tie_rng = np.random.default_rng(
        np.random.SeedSequence([SEED, int(tie_seed), int(user_id)])
    )
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(row[eligible]))))

    adjusted = row.copy()
    adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

    candidates = np.argpartition(adjusted, -K_EVAL)[-K_EVAL:]
    return candidates[np.argsort(adjusted[candidates])[::-1]].astype(np.int32)

# ============================================================
# 3. Generate baseline and gated recommendations
# ============================================================
tail_candidate_ids = np.flatnonzero(
    warm_tail_mask_tv & item_has_als_factors_tv
)

# gate -> tie seed -> user -> recommended item IDs
test_recs = {
    gate: {seed: {} for seed in TIE_SEEDS}
    for gate in GATES
}

for start in range(0, len(test_users), BATCH_SIZE):
    batch_users = np.asarray(
        test_users[start:start + BATCH_SIZE], dtype=np.int64
    )

    als_batch = score_als_batch(batch_users)
    content_batch = score_content_batch(batch_users)

    for row_idx, user_id_value in enumerate(batch_users):
        user_id = int(user_id_value)
        als_scores = als_batch[row_idx]

        # Candidate ranking uses content-v1's full-catalogue scores,
        # excluding the user's complete pre-test history.
        content_row = np.asarray(
            content_batch[row_idx], dtype=np.float64
        ).copy()

        # Preserve the established fallback for users with no positive history.
        if not u_has_pos_tv[user_id]:
            content_row = popularity_scores_tv.copy()

        content_row[seen_tv[user_id].indices] = -np.inf

        valid_tail = tail_candidate_ids[
            np.isfinite(content_row[tail_candidate_ids])
        ]

        if len(valid_tail):
            n_top = min(K_EVAL + 1, len(valid_tail))
            tail_scores = content_row[valid_tail]
            top_idx = np.argpartition(tail_scores, -n_top)[-n_top:]
            top_order = valid_tail[
                top_idx[np.argsort(tail_scores[top_idx])[::-1]]
            ]
        else:
            top_order = np.array([], dtype=np.int64)

        for seed in TIE_SEEDS:
            base = final_top_k(als_scores, user_id, seed)
            test_recs[0][seed][user_id] = base

            base_set = set(map(int, base))

            # Pick the highest-scoring warm-tail candidate not already in ALS@10.
            candidate = next(
                (
                    int(item) for item in top_order
                    if int(item) not in base_set
                ),
                None,
            )

            if candidate is None:
                candidate_rank = None
            else:
                candidate_score = content_row[candidate]
                candidate_rank = 1 + int(
                    np.sum(
                        np.isfinite(content_row)
                        & (content_row > candidate_score)
                    )
                )

            for gate in [500, 1000]:
                if (
                    candidate is not None
                    and candidate_rank is not None
                    and candidate_rank <= gate
                ):
                    gated = np.concatenate(
                        [base[:-1], np.array([candidate], dtype=np.int32)]
                    )
                else:
                    gated = base.copy()

                test_recs[gate][seed][user_id] = gated

# ============================================================
# 4. Evaluate per-user metrics at each seed and scope
# ============================================================
DISCOUNTS = 1.0 / np.log2(np.arange(2, K_EVAL + 2))

def evaluate_one_user(rec, truth):
    rec = np.asarray(rec, dtype=int)
    truth = set(truth)

    hits = np.array([int(item in truth) for item in rec], dtype=float)
    dcg = float(np.sum(hits * DISCOUNTS))

    ideal_hits = min(len(truth), K_EVAL)
    idcg = float(np.sum(DISCOUNTS[:ideal_hits])) if ideal_hits else 0.0

    return {
        "ndcg@10": dcg / idcg if idcg else 0.0,
        "recall@10": float(hits.sum() / len(truth)) if truth else 0.0,
        "hit_rate@10": float(hits.sum() > 0),
        "hit_count@10": float(hits.sum()),
    }

metric_rows = []
exposure_rows = []

for gate in GATES:
    for seed in TIE_SEEDS:
        recs_for_seed = test_recs[gate][seed]

        # Overall metrics: users with at least one relevant test target.
        for user_id, truth in test_relevant["overall"].items():
            metrics = evaluate_one_user(recs_for_seed[user_id], truth)
            metric_rows.append({
                "gate": gate,
                "seed": seed,
                "scope": "overall",
                "user_id": user_id,
                **metrics,
            })

            exposure_rows.append({
                "gate": gate,
                "seed": seed,
                "user_id": user_id,
                "warm_tail_slot_fraction": float(
                    warm_tail_mask_tv[recs_for_seed[user_id]].mean()
                ),
            })

        # Warm-tail metrics: only users with at least one warm-tail target.
        for user_id, truth in test_relevant["warm_tail"].items():
            metrics = evaluate_one_user(recs_for_seed[user_id], truth)
            metric_rows.append({
                "gate": gate,
                "seed": seed,
                "scope": "warm_tail",
                "user_id": user_id,
                **metrics,
            })

metrics_by_seed = pd.DataFrame(metric_rows)

# Average each user's results across tie-break seeds before aggregating users.
per_user_test = (
    metrics_by_seed
    .groupby(["gate", "scope", "user_id"], as_index=False)
    [["ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"]]
    .mean()
)

test_summary = (
    per_user_test
    .groupby(["gate", "scope"])
    [["ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"]]
    .mean()
    .round(6)
)

exposure_by_user = (
    pd.DataFrame(exposure_rows)
    .groupby(["gate", "user_id"])["warm_tail_slot_fraction"]
    .mean()
    .groupby("gate")
    .mean()
    .rename("warm_tail_recommendation_slot_fraction")
)

print("\n=== FINAL TEST METRICS ===")
display(test_summary)

print("\n=== Warm-tail recommendation exposure ===")
display(exposure_by_user.to_frame().round(6))

# ============================================================
# 5. Paired user-bootstrap confidence intervals vs ALS baseline
# ============================================================
rng_boot = np.random.default_rng(2026)
N_BOOT = BOOTSTRAP_REPS
BOOT_BATCH = 100

def paired_bootstrap_ci(differences):
    values = np.asarray(differences, dtype=np.float64)
    n = len(values)
    boot_means = np.empty(N_BOOT, dtype=np.float64)

    for start in range(0, N_BOOT, BOOT_BATCH):
        count = min(BOOT_BATCH, N_BOOT - start)
        idx = rng_boot.integers(0, n, size=(count, n))
        boot_means[start:start + count] = values[idx].mean(axis=1)

    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    return float(values.mean()), float(lo), float(hi)

ci_rows = []
ci_specs = [
    ("overall", "ndcg@10", "Overall NDCG@10 change"),
    ("overall", "recall@10", "Overall Recall@10 change"),
    ("warm_tail", "recall@10", "Warm-tail Recall@10 change"),
    ("warm_tail", "hit_rate@10", "Warm-tail HitRate@10 change"),
]

indexed = per_user_test.set_index(["gate", "scope", "user_id"])

for gate in [500, 1000]:
    for scope, col, label in ci_specs:
        baseline_users = indexed.loc[(0, scope)]
        gated_users = indexed.loc[(gate, scope)]

        common_users = baseline_users.index.intersection(gated_users.index)
        delta = (
            gated_users.loc[common_users, col].to_numpy()
            - baseline_users.loc[common_users, col].to_numpy()
        )

        mean, low, high = paired_bootstrap_ci(delta)
        ci_rows.append({
            "gate": gate,
            "metric": label,
            "mean_change": mean,
            "ci95_low": low,
            "ci95_high": high,
            "users": len(common_users),
        })

test_ci = pd.DataFrame(ci_rows)

print("\n=== PAIRED 95% USER-BOOTSTRAP CIs VS ALS ===")
display(test_ci.round(6))

print(
    "\nEvaluation complete. Test results were used only for this final assessment; "
    "do not use them to adjust the rank gate."
)


=== Locked test targets ===
Post-cutoff positive user-movie pairs: 26658
overall: 26,658 relevant pairs across 5,831 users
tail: 1,885 relevant pairs across 1,140 users
warm_tail: 1,810 relevant pairs across 1,121 users
strict_cold: 75 relevant pairs across 73 users

=== FINAL TEST METRICS ===


ndcg@10  recall@10  hit_rate@10  hit_count@10
gate scope                                                    
0    overall    0.076251   0.097556     0.302864      0.418796
     warm_tail  0.000000   0.000000     0.000000      0.000000
500  overall    0.076092   0.097151     0.302864      0.417424
     warm_tail  0.001685   0.004891     0.009813      0.009813
1000 overall    0.075069   0.094994     0.297548      0.406963
     warm_tail  0.002101   0.006230     0.011597      0.011597


=== Warm-tail recommendation exposure ===


,warm_tail_recommendation_slot_fraction
gate,
0,0.000000
500,0.022072
1000,0.054279



=== PAIRED 95% USER-BOOTSTRAP CIs VS ALS ===


,gate,metric,mean_change,ci95_low,ci95_high,users
0,500,Overall NDCG@10 change,-0.000159,-0.000399,0.000071,5831
1,500,Overall Recall@10 change,-0.000405,-0.001092,0.000199,5831
2,500,Warm-tail Recall@10 change,0.004891,0.001858,0.008727,1121
3,500,Warm-tail HitRate@10 change,0.009813,0.004460,0.016057,1121
4,1000,Overall NDCG@10 change,-0.001182,-0.001568,-0.000801,5831
5,1000,Overall Recall@10 change,-0.002563,-0.003636,-0.001535,5831
6,1000,Warm-tail Recall@10 change,0.006230,0.002661,0.010645,1121
7,1000,Warm-tail HitRate@10 change,0.011597,0.005352,0.017841,1121



Evaluation complete. Test results were used only for this final assessment; do not use them to adjust the rank gate.


In [48]:
from pathlib import Path
import json

reports = Path("../reports")
reports.mkdir(exist_ok=True)

test_summary.reset_index().to_csv(
    reports / "reserved_slot_test_metrics.csv", index=False
)

test_ci.to_csv(
    reports / "reserved_slot_test_paired_ci.csv", index=False
)

exposure_by_user.rename_axis("gate").reset_index().to_csv(
    reports / "reserved_slot_test_exposure.csv", index=False
)

test_config = {
    "training_data": "train + validation",
    "test_split": "sample_10pct test",
    "cutoff_utc": "2016-01-01T00:00:00Z",
    "minimum_relevant_rating": 4.0,
    "warm_tail_definition": "1-19 positive interactions",
    "selected_gate": 500,
    "secondary_gate": 1000,
    "recommendation_k": 10,
    "tie_break_seeds": 10,
    "bootstrap_replicates": 3000,
    "bootstrap_seed": 2026
}

with open(reports / "reserved_slot_test_config.json", "w") as f:
    json.dump(test_config, f, indent=4)

print("Final test artifacts exported.")

Final test artifacts exported.
